<a href="https://colab.research.google.com/github/eldhosekroy/churn_prediction/blob/main/better_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [426]:
import os
import re
import time
import nltk
import pickle
import random
import warnings

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.base import clone
from sklearn.utils import class_weight
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_curve, RocCurveDisplay, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

### **Directory Management**

In [427]:
output_dir = './output'

ARTIFACTS_PATH = output_dir
os.makedirs(ARTIFACTS_PATH, exist_ok=True)
print(f"Artifacts will be saved to: {ARTIFACTS_PATH}")

Artifacts will be saved to: ./output


# **Data Collection**

*   Enrolled dataset
*   All notes dataset





In [361]:
# Load 'All Notes-Contacts_copy.csv'
notes = pd.read_csv('/content/All Notes-Contacts_copy.csv')
print('\nNotes DataFrame Head:')
print(notes.head())


Notes DataFrame Head:
                   Note Id            Note Owner.id      Note Owner  \
0  zcrm_560042000001333918  zcrm_560042000001286184  SalesPerson-21   
1  zcrm_560042000001333864  zcrm_560042000001286184  SalesPerson-21   
2  zcrm_560042000001333760  zcrm_560042000001286184  SalesPerson-21   
3  zcrm_560042000001370043  zcrm_560042000001286184  SalesPerson-21   
4  zcrm_560042000001333945  zcrm_560042000001286184  SalesPerson-21   

  Note Title                                  Note Content  \
0        NaN                       SIJINA\nANOTHER CALLING   
1        NaN  SONU\nDATA ANALYTICS ALREADY INTERNSHIP DONE   
2        NaN                                   JOINED WORK   
3        NaN                           currently doing job   
4        NaN                         SHILKA\nNOT CONNECTED   

              Parent ID.id Parent ID            Created By.id Created By  \
0  zcrm_560042000000424058    sijina  zcrm_560042000001286184  Udhya P U   
1  zcrm_56004200000042405

In [362]:
# Load 'Endrolled & registred.csv'
enrolled_df = pd.read_csv('/content/Endrolled & registred.csv', encoding='latin1')
print('\nEnrolled & Registered DataFrame Head:')
print(enrolled_df.head())


Enrolled & Registered DataFrame Head:
                Contact Id         Contact Owner.id  Contact Owner First Name  \
0  zcrm_560042000000440092  zcrm_560042000000283001  SalesPerson-2        NaN   
1  zcrm_560042000000466061  zcrm_560042000000283001  SalesPerson-2        NaN   
2  zcrm_560042000000565094  zcrm_560042000000331113  SalesPerson-9       Alan   
3  zcrm_560042000000583691  zcrm_560042000000331113  SalesPerson-9        NaN   
4  zcrm_560042000000604837  zcrm_560042000000283001  SalesPerson-2        NaN   

            Last Name  Email Opt Out college Name.id college Name  \
0  AJSAL Mohammed T S          False             NaN          NaN   
1        Abhishek R J          False             NaN          NaN   
2              Thomas          False             NaN          NaN   
3             Abhinav          False             NaN          NaN   
4         Albin Shaji          False             NaN          NaN   

    Track Interested       Tag  ... Gender Year of Graduati

# **Data Handling**

In [363]:
# Columns of enrolled dataset
print(enrolled_df.columns)

Index(['Contact Id', 'Contact Owner.id', 'Contact Owner', 'First Name',
       'Last Name', 'Email Opt Out', 'college Name.id', 'college Name',
       'Track Interested', 'Tag', 'Description', 'Created By.id',
       'Modified By.id', 'Created Time', 'Modified Time', 'Last Activity Time',
       'Unsubscribed Mode', 'Unsubscribed Time', 'Contact Name', 'City',
       'Mailing State', 'Mailing Zip', 'Mailing Country', 'Program Joined.id',
       'Program Joined', 'Source of lead', 'Pipeline owner.id', 'Semester',
       'Batch Assigned to', 'Lead Generated on', 'Course', 'Gender',
       'Year of Graduation', 'Experience', 'Test', 'Followup Email', 'Invoice',
       'Mode of Program Joined', 'Program Location', 'Region',
       'Whatsapp Number'],
      dtype='object')


In [364]:
# Remove irrelevant columns
enrolled_df=enrolled_df.drop(columns=['First Name', 'Last Name', 'Email Opt Out', 'college Name.id', 'college Name', 'Description', 'Unsubscribed Mode', 'Unsubscribed Time', 'Mailing Zip', 'Batch Assigned to', 'Region'])
print(enrolled_df.columns)

Index(['Contact Id', 'Contact Owner.id', 'Contact Owner', 'Track Interested',
       'Tag', 'Created By.id', 'Modified By.id', 'Created Time',
       'Modified Time', 'Last Activity Time', 'Contact Name', 'City',
       'Mailing State', 'Mailing Country', 'Program Joined.id',
       'Program Joined', 'Source of lead', 'Pipeline owner.id', 'Semester',
       'Lead Generated on', 'Course', 'Gender', 'Year of Graduation',
       'Experience', 'Test', 'Followup Email', 'Invoice',
       'Mode of Program Joined', 'Program Location', 'Whatsapp Number'],
      dtype='object')


###### Handling 'Track Interested'

In [365]:
# Define the specific list of program names to use for filling nulls
program_names = ['DA', 'DS', 'MERN', 'Python', 'Gen AI', 'Prompt Engineering']

# Fill null values in 'Track Interested' with random choices from the program_names list
if program_names:
    # Only fill nulls where they exist in the 'Track Interested' column
    null_indices = enrolled_df['Track Interested'].isnull()
    num_nulls = null_indices.sum()
    if num_nulls > 0:
        enrolled_df.loc[null_indices, 'Track Interested'] = np.random.choice(program_names, size=num_nulls)
    print("Null values in 'Track Interested' after filling:")
    print(enrolled_df['Track Interested'].isnull().sum())
    print("Value counts for 'Track Interested' after filling (top 10):")
    print(enrolled_df['Track Interested'].value_counts().head(10))
else:
    print("Warning: The list of program names for 'Track Interested' is empty.")

Null values in 'Track Interested' after filling:
0
Value counts for 'Track Interested' after filling (top 10):
Track Interested
Data Analytics        211
Datascience           189
Python                131
Prompt Engineering    121
MERN                  121
Gen AI                121
DS                    114
DA                    113
Data Science           29
Dataanalytics          29
Name: count, dtype: int64


In [366]:
print("\nValue counts for 'Track Interested' *before* standardization:")
print(enrolled_df['Track Interested'].value_counts()) # Print top 20 to see diverse values


Value counts for 'Track Interested' *before* standardization:
Track Interested
Data Analytics                         211
Datascience                            189
Python                                 131
Prompt Engineering                     121
MERN                                   121
Gen AI                                 121
DS                                     114
DA                                     113
Data Science                            29
Dataanalytics                           29
MERN Stack                              20
Data Analytics Intern                   13
Data Analyst                            10
Not Mentioned                            9
Python Full stack                        8
PEP                                      5
Datascience/DataAnalytics                5
Data Analyst Intern                      4
Dataanalyst                              3
DataAnalytics                            3
Data Analytics Fresher                   2
Dataanlytics     

In [367]:
# standardize values in 'Track Interested'
def standardize_track_interested(track_string):
    if pd.isna(track_string):
        return 'Other Course'

    track_string = str(track_string).strip().upper()

    if 'DATA SCIENCE' in track_string or 'DATASCIENCE' in track_string or 'DS' in track_string:
        return 'Data Science'
    elif 'DATA ANALYTICS' in track_string or 'DATAANALYTICS' in track_string or 'DATA ANALYST' in track_string or 'DATAANALYST' in track_string or 'DA' in track_string:
        return 'Data Analytics'
    elif 'MERN' in track_string:
        return 'MERN Stack'
    elif 'PYTHON FULL STACK' in track_string:
        return 'Full Stack'
    elif 'PYTHON' in track_string:
        return 'Python Programming'
    elif 'GEN AI' in track_string or 'GENAI' in track_string or 'PROMPT ENGINEERING' in track_string:
        return 'Gen AI & Prompt Engineering'
    elif 'FULL STACK' in track_string or 'FS' in track_string:
        return 'Full Stack Development'
    elif 'DIGITAL MKT' in track_string or 'DIGITAL MARKETING' in track_string:
        return 'Digital Marketing'
    elif 'CLOUD' in track_string:
        return 'Cloud Computing'
    elif 'UI UX' in track_string:
        return 'UI/UX Design'
    elif 'AIML' in track_string or 'AI/ML' in track_string:
        return 'AI/ML'
    elif 'ADVANCED PROGRAM' in track_string:
        return 'Advanced Program'
    elif 'ONE MONTH PROGRAM' in track_string:
        return 'One Month Program'
    elif 'PEP PROGRAM' in track_string or 'PEP' in track_string:
        return 'PEP Program'
    elif 'NOT MENTIONED' in track_string:
        return 'Not Mentioned'

    return 'Other Course'

# Apply the standardization function to the 'Track Interested' column
enrolled_df['Track Interested'] = enrolled_df['Track Interested'].apply(standardize_track_interested)

print("\nValue counts for 'Track Interested' after standardization:")
print(enrolled_df['Track Interested'].value_counts().head(10))


Value counts for 'Track Interested' after standardization:
Track Interested
Data Analytics                 394
Data Science                   339
Gen AI & Prompt Engineering    245
MERN Stack                     141
Python Programming             131
Not Mentioned                    9
Full Stack                       8
PEP Program                      5
Other Course                     2
Name: count, dtype: int64


###### Handling 'Tag'

In [368]:
# Unique values in tag column
print(enrolled_df['Tag'].value_counts())

Tag
Ayisha                                      185
Andriya                                      57
Enrolled                                     45
April 25 Batch                               22
Mithlaj Records                              20
Future Campaign-April 24                      9
DA Feb 2025                                   9
sangeetha                                     9
Registered                                    7
Free Internship                               5
MES Kalladi                                   5
Leads                                         3
Ayisha,sangeetha                              2
Future Campaign-April 24,Mithlaj Records      2
Techware-Walk-in                              2
Old CRM data                                  2
Old FUP                                       2
Amal                                          2
Followup                                      1
Workshop on GEN Ai                            1
FAHEEM REF                          

In [369]:
# Get unique non-null values from the 'Tag' column
existing_tags = enrolled_df['Tag'].dropna().unique().tolist()

# Add 'Not Mentioned' to the list of possible tags
if 'Not Mentioned' not in existing_tags:
    existing_tags.append('Not Mentioned')

# Identify null indices in the 'Tag' column
null_indices = enrolled_df['Tag'].isnull()
num_nulls = null_indices.sum()

# Fill null values with random choices from the combined list
if num_nulls > 0 and existing_tags:
    enrolled_df.loc[null_indices, 'Tag'] = np.random.choice(existing_tags, size=num_nulls)
    print("Null values in 'Tag' after filling:")
    print(enrolled_df['Tag'].isnull().sum())
    print("\nValue counts for 'Tag' after filling:")
    print(enrolled_df['Tag'].value_counts())
elif num_nulls == 0:
    print("No null values found in 'Tag' column.")
else:
    print("Warning: The list of existing tags is empty, or no nulls to fill.")

Null values in 'Tag' after filling:
0

Value counts for 'Tag' after filling:
Tag
Ayisha                                      208
Andriya                                      91
Enrolled                                     73
Mithlaj Records                              49
April 25 Batch                               42
Leads                                        41
sangeetha                                    40
Amal                                         38
Gokul                                        37
Registered                                   35
Free Internship                              34
DA Feb 2025                                  34
MES Kalladi                                  34
SHE REEBOOT                                  33
Workshop on GEN Ai                           32
FAHEEM REF                                   32
Future Campaign-April 24                     31
hot                                          31
without pipelines 18/02                      29
Jeevan 

###### Handling 'City'

In [370]:
# Unique values in 'city'
print(enrolled_df['City'].value_counts().head(20))

City
Ernakulam             57
Thrissur              32
Kottayam              19
Malappuram            15
Kozhikode              9
Kollam                 9
Kannur                 9
Pathanamthitta         7
Kochi, Kerala          7
Wayanad                6
Alappuzha              6
Palakkad               5
Malappuram, Kerala     4
Idukki                 4
Kerala                 4
Thiruvananthapuram     4
Others                 3
Calicut, Kerala        3
Ernakulam, Kerala      2
Kochi                  2
Name: count, dtype: int64


In [371]:
# Get unique non-null values from the 'City' column
existing_cities = enrolled_df['City'].dropna().unique().tolist()

# Identify null indices in the 'City' column
null_indices = enrolled_df['City'].isnull()
num_nulls = null_indices.sum()

# Fill null values with random choices from the existing cities
if num_nulls > 0 and existing_cities:
    enrolled_df.loc[null_indices, 'City'] = np.random.choice(existing_cities, size=num_nulls)
    print("Null values in 'City' after filling:")
    print(enrolled_df['City'].isnull().sum())
    print("\nValue counts for 'City' after filling:")
    print(enrolled_df['City'].value_counts())
elif num_nulls == 0:
    print("No null values found in 'City' column.")
else:
    print("Warning: The list of existing cities is empty, or no nulls to fill.")

Null values in 'City' after filling:
0

Value counts for 'City' after filling:
City
Ernakulam                         74
Thrissur                          42
Malappuram                        38
Kottayam                          30
Calicut, Kerala                   29
                                  ..
Calicut                           14
Thazhvettippuram, P.O, Kerala,    14
Bengaluru,                        13
Kerala, India                     13
Vaikam, Kerala                    13
Name: count, Length: 64, dtype: int64


In [372]:
def standardize_city_name(city_string):
    if pd.isna(city_string):
        return 'Unknown'

    city_string = str(city_string).strip().upper()

    # Common standardizations
    city_string = re.sub(r'[,\s]*KERALA[\s\.]*', '', city_string) # Remove 'Kerala'
    city_string = re.sub(r'[,\s]*INDIA[\s\.]*', '', city_string) # Remove 'India'
    city_string = re.sub(r'[,\s]*IN[\s\.]*', '', city_string) # Remove 'IN'

    # Consolidate common city names
    if 'ERNAKULAM' in city_string or 'KOCHI' in city_string or 'MUVATUPUZHA' in city_string or 'KOTHAMANGALAM' in city_string or 'PADHUVAPURAM' in city_string:
        return 'Ernakulam'
    elif 'THRISSUR' in city_string or 'TRICHUR' in city_string or 'POTTA' in city_string or 'CHERUVALLOOR' in city_string:
        return 'Thrissur'
    elif 'MALAPPURAM' in city_string or 'PERTALMANNA' in city_string:
        return 'Malappuram'
    elif 'KOTTAYAM' in city_string or 'VAIKAM' in city_string or 'KANJIRAPPALLY' in city_string:
        return 'Kottayam'
    elif 'KOZHIKODE' in city_string or 'CALICUT' in city_string or 'VADAKARA' in city_string or 'SARADHALAYAM SIDHASAMAJAM' in city_string:
        return 'Calicut'
    elif 'THIRUVANANTHAPURAM' in city_string or 'TRIVANDRUM' in city_string:
        return 'Trivandrum'
    elif 'KOLLAM' in city_string or 'KOTTARAKARA' in city_string:
        return 'Kollam'
    elif 'PALAKKAD' in city_string:
        return 'Palakkad'
    elif 'KANNUR' in city_string:
        return 'Kannur'
    elif 'ALAPPUZHA' in city_string or 'ALLEPPEY' in city_string:
        return 'Alappuzha'
    elif 'WAYANAD' in city_string:
        return 'Wayanad'
    elif 'PATHANAMTHITTA' in city_string or 'PANDALAM' in city_string or 'THAZHVETTIPPURAM' in city_string:
        return 'Pathanamthitta'
    elif 'OTTAPALAM' in city_string or 'PALAKKAD' in city_string:
        return 'Palakkad'
    elif 'IDUKKI' in city_string or 'THODUPUZHA' in city_string or 'IDUKKY' in city_string:
        return 'Idukki'
    elif 'KASARGODE' in city_string or 'KASARGOD' in city_string:
        return 'Kasargod'
    elif 'CHENNAI' in city_string:
        return 'Chennai'
    elif 'BENGALURU' in city_string or 'BANGALORE' in city_string:
        return 'Bengaluru'
    elif 'HYDERABAD' in city_string:
        return 'Hyderabad'
    elif 'MUMBAI' in city_string or 'BOMBAY' in city_string:
        return 'Mumbai'
    elif 'DELHI' in city_string:
        return 'Delhi'

    # Remove specific noise after checking major cities
    city_string = re.sub(r'\b(TOWNSHIP|TOWN|DISTRICT|DT)\b', '', city_string).strip()
    city_string = re.sub(r'\b(POST|PO)\b', '', city_string).strip()
    city_string = re.sub(r'\s*\d{6}\s*', '', city_string) # Remove postal codes

    # Handle specific cases that might still be left after initial broader checks
    if city_string == 'OTHERS' or city_string == 'UNKNOWN' or city_string == '-' or city_string == 'TREESA SEBASTIAN':
        return 'Unknown'
    if city_string == 'KOOVAPPADY':
        return 'Ernakulam' # Assuming Koovappady is near Ernakulam

    # If the string is empty after cleaning, return 'Unknown'
    if not city_string:
        return 'Unknown'

    # Default to 'Other' if not specifically matched, to consolidate less frequent cities
    #if enrolled_df['City'].value_counts()[city_string] < 5: # Example threshold, can be adjusted
    #    return 'Other'

    return city_string.title() # Convert to title case for consistency


enrolled_df['City'] = enrolled_df['City'].apply(standardize_city_name)

print("\nValue counts for 'City' after standardization:")
print(enrolled_df['City'].value_counts().head(20))


Value counts for 'City' after standardization:
City
Ernakulam         283
Thrissur          136
Calicut           120
Kottayam           92
Unknown            81
Malappuram         81
Alappuzha          73
Pathanamthitta     68
Idukki             66
Kollam             60
Palakkad           53
Trivandrum         45
Kannur             38
Wayanad            34
Chennai            16
Kasargod           15
Bengaluru          13
Name: count, dtype: int64


###### Handling 'Mailing State'

In [373]:
# Unique values in 'Mailing State'
print(enrolled_df['Mailing State'].value_counts())

Mailing State
Kerala           7
689647           1
MUVATTUPUZHA.    1
Name: count, dtype: int64


In [374]:
def standardize_mailing_state(state_string):
    if pd.isna(state_string):
        return np.nan # Keep as NaN for now to fill later

    state_string = str(state_string).strip().upper()

    if 'KERALA' in state_string or 'MUVATTUPUZHA.' in state_string or '689647' in state_string:
        return 'Kerala'
    elif 'TAMIL NADU' in state_string or 'CHENNAI' in state_string:
        return 'Tamil Nadu'
    elif 'KARNATAKA' in state_string or 'BENGALURU' in state_string or 'BANGALORE' in state_string:
        return 'Karnataka'
    else:
        return 'Other State'

# Apply the initial standardization
enrolled_df['Mailing State'] = enrolled_df['Mailing State'].apply(standardize_mailing_state)

def fill_mailing_state_from_city(row):
    current_state = row['Mailing State']
    city = row['City']

    # If the state is still NaN or 'Other State', try to infer from city
    if pd.isna(current_state) or current_state == 'Other State':
        if pd.isna(city) or city == 'Unknown' or city == 'Other':
            return 'Other State'

        # Map cities to states based on prior standardization
        if city in ['Ernakulam', 'Thrissur', 'Calicut', 'Kottayam', 'Malappuram',
                    'Alappuzha', 'Pathanamthitta', 'Idukki', 'Kollam', 'Trivandrum',
                    'Kannur', 'Wayanad', 'Palakkad', 'Kasargod']:
            return 'Kerala'
        elif city == 'Chennai':
            return 'Tamil Nadu'
        elif city == 'Bengaluru':
            return 'Karnataka'
        else:
            return 'Other State'
    return current_state

# Apply the filling function
enrolled_df['Mailing State'] = enrolled_df.apply(fill_mailing_state_from_city, axis=1)

print("\nValue counts for 'Mailing State' after standardization and filling:")
print(enrolled_df['Mailing State'].value_counts())


Value counts for 'Mailing State' after standardization and filling:
Mailing State
Kerala         1165
Other State      80
Tamil Nadu       16
Karnataka        13
Name: count, dtype: int64


###### Handling 'Mailing Country'

In [375]:
# Unique values in 'Mailing Country'
print(enrolled_df['Mailing Country'].value_counts())

Mailing Country
India    8
Name: count, dtype: int64


In [376]:
# Fill null values with 'India'
enrolled_df['Mailing Country'] = enrolled_df['Mailing Country'].fillna('India')

print("\nValue counts for 'Mailing Country' after filling nulls:")
print(enrolled_df['Mailing Country'].value_counts())


Value counts for 'Mailing Country' after filling nulls:
Mailing Country
India    1274
Name: count, dtype: int64


###### Handling 'Source of lead'

In [377]:
# Fill null values with mode of the column
if enrolled_df['Source of lead'].isnull().any():
    mode_source_of_lead = enrolled_df['Source of lead'].mode()[0]
    enrolled_df['Source of lead'] = enrolled_df['Source of lead'].fillna(mode_source_of_lead)
    print(f"Null values in 'Source of lead' filled with: {mode_source_of_lead}")
else:
    print("No null values found in 'Source of lead' to fill.")

print("\nValue counts for 'Source of lead' after filling nulls:")
print(enrolled_df['Source of lead'].value_counts())

Null values in 'Source of lead' filled with: Indeed

Value counts for 'Source of lead' after filling nulls:
Source of lead
Indeed                          420
Reference                       202
Digital Marketing (Goat add)    191
Infopark website                142
Enquiry                          78
Digital Marketing (Add On)       51
DM                               30
IV                               22
Workshop                         22
Seminar                          19
dm                               18
Website Enquiry                  16
infopark                         10
Naukri Paid                       8
Free internship campaign          7
Webinar                           5
Job Fair                          4
Social media                      4
Indeed Paid                       3
Carryover                         3
Infopark                          3
OLX                               3
Naukri Ad posting                 2
SEO                               2
Bulk Lead (30

In [378]:
#Standardize the values in the 'source of lead'
def standardize_source_of_lead(source_string):
    if pd.isna(source_string):
        return 'Unknown'

    source_string = str(source_string).strip().upper()

    if 'INDEED' in source_string:
        return 'Indeed'
    elif 'DIGITAL MARKETING' in source_string or 'DM' in source_string:
        return 'Digital Marketing'
    elif 'INFOPARK WEBSITE' in source_string or 'INFOPARK' in source_string:
        return 'Infopark Website'
    elif 'REFERENCE' in source_string or 'REFERAL' in source_string or 'REFERALS' in source_string:
        return 'Referral'
    elif 'ENQUIRY' in source_string or 'WEBSITE ENQUIRY' in source_string:
        return 'Website'
    elif 'FREE INTERNSHIP' in source_string:
        return 'Free Internship Campaign'
    elif 'NAUKRI' in source_string:
        return 'Naukri'
    elif 'WORKSHOP' in source_string:
        return 'Workshop'
    elif 'SEMINAR' in source_string:
        return 'Seminar'
    elif 'WEBINAR' in source_string:
        return 'Webinar'
    elif 'JOB FAIR' in source_string:
        return 'Job Fair'
    elif 'SOCIAL MEDIA' in source_string:
        return 'Social Media'
    elif 'LINKEDIN' in source_string:
        return 'LinkedIn'
    elif 'IV' in source_string:
        return 'IV'
    elif 'CARRYOVER' in source_string:
        return 'Carryover'
    elif 'OLX' in source_string:
        return 'OLX'
    elif 'SEO' in source_string:
        return 'SEO'
    elif 'BULK LEAD' in source_string:
        return 'Bulk Lead'
    elif 'OTHERS' in source_string:
        return 'Other'

    return source_string.title() # Default to Title Case if not explicitly matched

enrolled_df['Source of lead'] = enrolled_df['Source of lead'].apply(standardize_source_of_lead)

print("\nValue counts for 'Source of lead' after standardization:")
print(enrolled_df['Source of lead'].value_counts())


Value counts for 'Source of lead' after standardization:
Source of lead
Indeed                      423
Digital Marketing           291
Referral                    204
Infopark Website            157
Website                      94
IV                           22
Workshop                     22
Seminar                      19
Naukri                       10
Free Internship Campaign      8
Webinar                       5
Job Fair                      4
Social Media                  4
OLX                           3
Carryover                     3
Bulk Lead                     2
SEO                           2
LinkedIn                      1
Name: count, dtype: int64


###### Handling 'Year of graduation'

In [379]:
# Unique values in 'year of graduation'
print(enrolled_df['Year of Graduation'].value_counts())

Year of Graduation
2024.0     115
2025.0      72
2023.0      32
2022.0      21
2020.0      11
2017.0      11
2021.0      10
2019.0       9
2018.0       7
2016.0       6
2015.0       4
2007.0       4
2009.0       4
2026.0       3
2013.0       2
2014.0       2
2011.0       2
2012.0       1
20224.0      1
2006.0       1
2010.0       1
Name: count, dtype: int64


In [380]:
# Correct the erroneous year '20224' to '2024'
enrolled_df['Year of Graduation'] = enrolled_df['Year of Graduation'].replace(20224.0, 2024.0)

# Identify null values
null_years_indices = enrolled_df['Year of Graduation'].isnull()
num_null_years = null_years_indices.sum()

# Fill null values with random years between 2015 and 2027 (inclusive)
if num_null_years > 0:
    # np.random.randint is exclusive of the high value, so use 2028 for 2027 inclusive
    random_years = np.random.randint(2015, 2028, size=num_null_years)
    enrolled_df.loc[null_years_indices, 'Year of Graduation'] = random_years
    print(f"Filled {num_null_years} null values in 'Year of Graduation' with random years between 2015 and 2027.")
else:
    print("No null values found in 'Year of Graduation' to fill.")

# Convert 'Year of Graduation' to integer type
enrolled_df['Year of Graduation'] = enrolled_df['Year of Graduation'].astype(int)

print("\nValue counts for 'Year of Graduation' after cleaning and type conversion:")
print(enrolled_df['Year of Graduation'].value_counts().sort_index())

Filled 955 null values in 'Year of Graduation' with random years between 2015 and 2027.

Value counts for 'Year of Graduation' after cleaning and type conversion:
Year of Graduation
2006      1
2007      4
2009      4
2010      1
2011      2
2012      1
2013      2
2014      2
2015     83
2016     85
2017     80
2018     77
2019     92
2020     82
2021     80
2022     99
2023    112
2024    184
2025    137
2026     70
2027     76
Name: count, dtype: int64


###### Handling 'Lead generated on'

In [381]:
# Get unique non-null values from the 'Lead Generated on' column
existing_lead_dates = enrolled_df['Lead Generated on'].dropna().unique().tolist()

# Identify null indices in the 'Lead Generated on' column
null_indices = enrolled_df['Lead Generated on'].isnull()
num_nulls = null_indices.sum()

# Fill null values with random choices from the existing dates
if num_nulls > 0 and existing_lead_dates:
    enrolled_df.loc[null_indices, 'Lead Generated on'] = np.random.choice(existing_lead_dates, size=num_nulls)
    print("Null values in 'Lead Generated on' after filling:")
    print(enrolled_df['Lead Generated on'].isnull().sum())
    print("\nValue counts for 'Lead Generated on' after filling:")
    print(enrolled_df['Lead Generated on'].value_counts())
elif num_nulls == 0:
    print("No null values found in 'Lead Generated on' column.")
else:
    print("Warning: The list of existing 'Lead Generated on' dates is empty, or no nulls to fill.")

Null values in 'Lead Generated on' after filling:
0

Value counts for 'Lead Generated on' after filling:
Lead Generated on
20-09-2024    10
13-12-2023    10
28-06-2025     9
22-11-2024     8
20-08-2024     8
              ..
02-03-2024     1
19-05-2024     1
24-08-2024     1
19-07-2025     1
25-12-2016     1
Name: count, Length: 442, dtype: int64


###### Handling 'Mode of program joined'

In [382]:
# Get unique non-null values from the 'Mode of Program Joined' column
existing_modes = enrolled_df['Mode of Program Joined'].dropna().unique().tolist()

# Identify null indices in the 'Mode of Program Joined' column
null_indices = enrolled_df['Mode of Program Joined'].isnull()
num_nulls = null_indices.sum()

# Fill null values with random choices from the existing values
if num_nulls > 0 and existing_modes:
    enrolled_df.loc[null_indices, 'Mode of Program Joined'] = np.random.choice(existing_modes, size=num_nulls)
    print("Null values in 'Mode of Program Joined' after filling:")
    print(enrolled_df['Mode of Program Joined'].isnull().sum())
    print("\nValue counts for 'Mode of Program Joined' after filling:")
    print(enrolled_df['Mode of Program Joined'].value_counts())
elif num_nulls == 0:
    print("No null values found in 'Mode of Program Joined' column.")
else:
    print("Warning: The list of existing 'Mode of Program Joined' values is empty, or no nulls to fill.")

Null values in 'Mode of Program Joined' after filling:
0

Value counts for 'Mode of Program Joined' after filling:
Mode of Program Joined
Onsite           533
Online           312
Hybrid           217
online hybrid    212
Name: count, dtype: int64


In [383]:
# Standardize the values in the 'mode of program joined'
def standardize_mode_of_program_joined(mode_string):
    if pd.isna(mode_string):
        return None
    mode_string = str(mode_string).strip().lower()
    if 'online' in mode_string and 'hybrid' not in mode_string:
        return 'Online'
    elif 'offline' in mode_string or 'onsite' in mode_string:
        return 'Offline'
    elif 'hybrid' in mode_string:
        return 'Hybrid'
    return None

enrolled_df['Mode of Program Joined'] = enrolled_df['Mode of Program Joined'].apply(standardize_mode_of_program_joined)

print("Value counts for 'Mode of Program Joined' after standardization:")
print(enrolled_df['Mode of Program Joined'].value_counts())

Value counts for 'Mode of Program Joined' after standardization:
Mode of Program Joined
Offline    533
Hybrid     429
Online     312
Name: count, dtype: int64


###### Handling 'program location'

In [384]:
# Unique values in the 'program location'
print(enrolled_df['Program Location'].value_counts())

Program Location
Kochi                247
online                95
Calicut               85
Hyderabad              2
Online                 2
Bengaluru              2
kochi live stream      1
banglore               1
kochi live             1
Rajahmundry            1
Bangalore              1
Name: count, dtype: int64


In [385]:
# Standardize values in the 'program location'
def standardize_program_location(location_string):
    if pd.isna(location_string):
        return None

    location_string = str(location_string).strip().lower()

    # Remove 'online' values
    if 'online' in location_string:
        return None

    # Standardize specific cities
    if 'kochi' in location_string:
        return 'Kochi'
    elif 'calicut' in location_string:
        return 'Calicut'
    elif 'bengaluru' in location_string or 'banglore' in location_string:
        return 'Bengaluru'
    elif 'hyderabad' in location_string:
        return 'Hyderabad'

    # All other values are categorized as 'Other Location'
    return None

enrolled_df['Program Location'] = enrolled_df['Program Location'].apply(standardize_program_location)

print("Value counts for 'Program Location' after standardization:")
print(enrolled_df['Program Location'].value_counts())

Value counts for 'Program Location' after standardization:
Program Location
Kochi        249
Calicut       85
Bengaluru      3
Hyderabad      2
Name: count, dtype: int64


In [386]:
# Fill null values in 'program location'
null_locations = enrolled_df['Program Location'].isnull()
num_nulls_location = null_locations.sum()

if num_nulls_location > 0:
    # Define locations and their weights for random filling
    locations_to_fill = ['Kochi', 'Calicut', 'Bengaluru', 'Hyderabad']
    weights = [0.45, 0.45, 0.05, 0.05] # Kochi and Calicut get 45% each, others 5%

    # Generate random choices based on weights
    filled_values = np.random.choice(locations_to_fill, size=num_nulls_location, p=weights)
    enrolled_df.loc[null_locations, 'Program Location'] = filled_values

    print(f"Filled {num_nulls_location} null values in 'Program Location' with weighted random choices.")
else:
    print("No null values found in 'Program Location' to fill.")

print("\nValue counts for 'Program Location' after filling nulls:")
print(enrolled_df['Program Location'].value_counts())

Filled 935 null values in 'Program Location' with weighted random choices.

Value counts for 'Program Location' after filling nulls:
Program Location
Kochi        664
Calicut      521
Hyderabad     45
Bengaluru     44
Name: count, dtype: int64


###### Handling 'Whatsapp Number'

In [387]:
# Values in 'whatsapp number'
print(enrolled_df['Whatsapp Number'].value_counts())

Whatsapp Number
907428451.0    1
808661817.0    1
701218892.0    1
974470699.0    1
Name: count, dtype: int64


In [388]:
# Identify null values in 'Whatsapp Number'
null_whatsapp_indices = enrolled_df['Whatsapp Number'].isnull()
num_null_whatsapp = null_whatsapp_indices.sum()

if num_null_whatsapp > 0:
    # Generate random 10-digit numbers for the nulls. Start with 6, 7, 8, or 9.
    generated_numbers = []
    for _ in range(num_null_whatsapp):
        first_digit = random.choice([6, 7, 8, 9])
        remaining_digits = ''.join([str(random.randint(0, 9)) for _ in range(9)])
        generated_numbers.append(float(str(first_digit) + remaining_digits))

    enrolled_df.loc[null_whatsapp_indices, 'Whatsapp Number'] = generated_numbers
    print(f"Filled {num_null_whatsapp} null values in 'Whatsapp Number' with random 10-digit numbers.")
else:
    print("No null values found in 'Whatsapp Number' to fill.")

# Convert 'Whatsapp Number' to integer type. Use errors='coerce' to handle any non-numeric data gracefully.
enrolled_df['Whatsapp Number'] = enrolled_df['Whatsapp Number'].astype(str).str.replace('.0', '', regex=False).astype(float).astype('Int64')

print("\nValue counts for 'Whatsapp Number' after filling nulls and type conversion (top 10):")
print(enrolled_df['Whatsapp Number'].value_counts().head(10))
print(f"\nData type of 'Whatsapp Number' column: {enrolled_df['Whatsapp Number'].dtype}")

Filled 1270 null values in 'Whatsapp Number' with random 10-digit numbers.

Value counts for 'Whatsapp Number' after filling nulls and type conversion (top 10):
Whatsapp Number
8391511202    1
7517994089    1
6604583106    1
7514559487    1
6913476532    1
6208425458    1
7784759764    1
7080987760    1
6095273418    1
6802428737    1
Name: count, dtype: Int64

Data type of 'Whatsapp Number' column: Int64


###### Handling 'Email ID'

In [389]:
# Create new column for email ids
generated_emails = set()

def generate_unique_email_id(contact_name):
    if pd.isna(contact_name):
        return None

    # Convert name to lowercase and remove non-alphanumeric characters for email prefix
    base_email = re.sub(r'[^a-z0-9]', '', str(contact_name).lower())

    if not base_email: # Handle cases where name might be empty after cleaning
        base_email = 'user'

    email_prefix = base_email
    counter = 1

    # Ensure uniqueness
    while f"{email_prefix}@gmail.com" in generated_emails:
        email_prefix = f"{base_email}{counter}"
        counter += 1

    final_email = f"{email_prefix}@gmail.com"
    generated_emails.add(final_email)
    return final_email

# Apply the function to create the new 'Email ID' column
enrolled_df['Email ID'] = enrolled_df['Contact Name'].apply(generate_unique_email_id)

print("First 5 generated Email IDs:")
print(enrolled_df['Email ID'].head())

print("\nNumber of unique Email IDs generated:")
print(enrolled_df['Email ID'].nunique())

print("Total number of candidates:")
print(len(enrolled_df))

First 5 generated Email IDs:
0    ajsalmohammedts@gmail.com
1         abhishekrj@gmail.com
2         alanthomas@gmail.com
3            abhinav@gmail.com
4         albinshaji@gmail.com
Name: Email ID, dtype: object

Number of unique Email IDs generated:
1274
Total number of candidates:
1274


###### Handling 'Invoice'

In [390]:
# Fill null values in 'Invoice' column with 'No'
enrolled_df['Invoice'] = enrolled_df['Invoice'].fillna('No')

# Standardize 'Invoice' column values to 'Yes' or 'No'
def standardize_invoice(invoice_status):
    if pd.isna(invoice_status):
        return 'No' # Should have been filled by previous step, but as a safeguard
    invoice_status = str(invoice_status).strip().lower()
    if invoice_status == 'yes':
        return 'Yes'
    else:
        return 'No'

enrolled_df['Invoice'] = enrolled_df['Invoice'].apply(standardize_invoice)

print("Value counts for 'Invoice' after filling nulls and standardization:")
print(enrolled_df['Invoice'].value_counts())

Value counts for 'Invoice' after filling nulls and standardization:
Invoice
Yes    1084
No      190
Name: count, dtype: int64


###### Handling 'Education'(course)

In [391]:
# Function to standardize course names in the 'Course' column, focusing on preserving degree names
def standardize_course_name(course_string):
    if pd.isna(course_string) or str(course_string).strip().lower() == 'not mentioned':
        return None

    course_string = str(course_string).upper()

    # Remove general noise: extra spaces, some punctuation, but be careful not to remove critical parts
    # Allow A-Z, 0-9, spaces, &, and hyphens. Only allow these characters
    course_string = re.sub(r'[^-A-Z0-9\s&]', '', course_string)
    course_string = ' '.join(course_string.split()) # Normalize spaces

    # Standardize common degree and course abbreviations
    course_string = course_string.replace('B.TECH', 'BTECH')
    course_string = course_string.replace('B.E.', 'BE')
    course_string = course_string.replace('M.TECH', 'MTECH')
    course_string = course_string.replace('BSC CS', 'BSC-CS')
    course_string = course_string.replace('MSC CS', 'MSC-CS')
    #course_string = course_string.replace('COMPUTER APPLICATION', 'CA')
    course_string = course_string.replace('BACHELOR OF COMPUTER APPLICATION - COMPUTER APPLICATION', 'BCA')
    # Add BVOC specific standardization
    course_string = course_string.replace('BVOC DATA ANALYTICS AND MACHINE LEARNING', 'B VOC-IT')
    course_string = course_string.replace('BVOC', 'B VOC') # General BVOC

    course_string = course_string.replace('COMPUTER SCIENCE AND ENGINEERING', 'BTECH')
    #course_string = course_string.replace('COMPUTER SCIENCE', 'BTECH')
    #course_string = course_string.replace('INFORMATION TECHNOLOGY', 'BTECH')
    course_string = course_string.replace('DATA ANALYTICS', 'DA')
    course_string = course_string.replace('DATA SCIENCE', 'DS')
    #course_string = course_string.replace('BIG DATA ANALYTICS', 'BDA')
    course_string = course_string.replace('ENGLISH LITERATURE', 'BA')
    course_string = course_string.replace('PLUS TWO', 'PLUS TWO')
    course_string = course_string.replace('DIPLOMA', 'DIPLOMA')
    course_string = course_string.replace('DIPLOMA-NON IT', 'DIPLOMA')

    # Handle common course name variations before general IT/NON-IT replacement
    course_string = course_string.replace('B TECH', 'BTECH') # For 'B TECH'
    course_string = course_string.replace('B_TECH', 'BTECH') # For 'B_TECH'
    course_string = course_string.replace('B_COM', 'BCOM') # For 'B_COM'
    course_string = course_string.replace('CA CA', 'CA')
    course_string = course_string.replace('CA-CA', 'CA') # For 'CA - CA' and 'CA CA'

    # Specific fixes for previous output issues (e.g., missing hyphens)
    course_string = course_string.replace('BTECHIT', 'BTECH')
    course_string = course_string.replace('MSCIT', 'MSC-IT')
    course_string = course_string.replace('BSCIT', 'BSC-IT')
    course_string = course_string.replace('MTECH-IT', 'MTECH')
    course_string = course_string.replace('MTECHIT', 'MTECH')

    # Address remaining specific MSC-CS-DA variations directly after initial replacements
    course_string = course_string.replace('MSC-CS DA', 'MSC-IT')
    course_string = course_string.replace('MSC-CSDA', 'MSC-IT')

    # These replacements should happen after primary acronym standardization
    # This line might create issues for standalone 'IT' if not careful. For now, keep as is as other specific replacements handle degree-IT combinations.
    course_string = course_string.replace(' IT', '-IT') # Handle BTech IT etc. to BTech-IT
    course_string = course_string.replace('NON-IT', '-NON-IT') # Handle BTech-Non IT etc.

    # Normalize hyphens: replace multiple hyphens with single, remove spaces around hyphens
    course_string = re.sub(r'-+', '-', course_string).strip('-')
    course_string = re.sub(r'\s*-\s*', '-', course_string) # Remove spaces around hyphens
    course_string = ' '.join(course_string.split()) # Final space normalization after hyphen cleanup

    # Special handling for numerical and very short entries
    if course_string == '2':
        return 'PLUS TWO'
    if course_string == 'COMP':
        return 'BCA'
    if course_string == 'POST GRADUATION':
        return 'PG'
    if course_string == 'BSC SCIENCE':
        return 'BSC'
    if course_string == 'GRADUATED':
        return 'GRADUATED'
    if course_string == 'DIPLOMA-IT':
        return 'DIPLOMA-IT'
    if course_string == 'DIPLOMA-NON-IT':
        return 'DIPLOMA'
    if course_string == 'DIPLOMA':
        return 'DIPLOMA'
    if course_string == 'INCOLLEGE' or course_string == 'IN_COLLEGE' or course_string == 'STUDENT': # Standardize IN_COLLEGE and convert 'STUDENT' if found as a course
        return 'UNSPECIFIED' # 'STUDENT' is a role, not a course
    if course_string == 'OTHERS':
        return None

    # Remove words like 'PROGRAM', 'DEGREE', 'BACHELOR', 'MASTER' if they are standalone or don't contribute to specific degree names
    # This part should be less aggressive
    course_string = re.sub(r'\b(BACHELOR|MASTER|DEGREE|PROGRAMME|PROGRAM)\b', '', course_string).strip()
    course_string = re.sub(r'\b(OF|AND)\b', '', course_string).strip() # Remove common connectors

    course_string = ' '.join(course_string.split()) # Final space normalization

    # Map common variations to desired output, e.g., 'BTECH-IT' or 'BTECH'
    if 'BTECH' in course_string and 'NON' not in course_string and 'IT' not in course_string:
        course_string = 'BTECH'
    if 'BCA' in course_string:
        course_string = 'BCA'
    if 'MCA' in course_string:
        course_string = 'MCA'
    if 'BCOM' in course_string:
        course_string = 'BCOM'
    if 'MCOM' in course_string:
        course_string = 'MCOM'
    if 'BA' in course_string and 'DA' not in course_string and 'MBA' not in course_string:
        course_string = 'BA'
    if 'MA' in course_string:
        course_string = 'MA'
    if 'BSC' in course_string and 'NON' not in course_string and 'IT' not in course_string and 'CS' not in course_string and 'COMPUTER SCIENCE' not in course_string:
        course_string = 'BSC'
    if 'MSC' in course_string and 'NON' not in course_string and 'IT' not in course_string and 'CS' not in course_string and 'COMPUTER SCIENCE' not in course_string and 'BIG' not in course_string:
        course_string = 'MSC'
    if 'CSE' in course_string:
        course_string = 'BTECH'
    if 'CS' in course_string and 'BSC' not in course_string and 'MSC' not in course_string:
        course_string = 'BTECH'
    if 'COMPUTER SCIENCE' in course_string and 'BSC' not in course_string and 'MSC' not in course_string:
        course_string = 'BTECH'
    if 'B VOC-IT' in course_string:
        course_string = 'B VOC-IT'
    if 'DIPLOMA'in course_string and 'IT' not in course_string:
        course_string = 'DIPLOMA'


    # Ensure consistency for entries like 'BTECH-IT', 'BSC-IT', etc.
    if course_string == 'BTECH-IT':
         course_string = 'BTECH'
    elif course_string == 'BTECH-NON-IT':
         course_string = 'BTECH'
    elif course_string == 'BACHELOR OF COMPUTER APPLICATION - COMPUTER APPLICATION':
         course_string = 'BCA'
    elif course_string == 'BSC-IT':
        pass # Keep as is
    elif course_string == 'MSC-IT':
        pass
    elif course_string == 'BSCSCIENCE':
         course_string = 'BSC'
    elif course_string == 'BSC COMPUTER SCIENCE':
         course_string = 'BSC-IT'
    elif course_string == 'MSC COMPUTER SCIENCEDA':
         course_string = 'MSC-IT'
    elif course_string == 'MSC COMPUTER SCIENCE DA':
         course_string = 'MSC-IT'
    elif course_string == 'MSC BIG DA':
         course_string = 'MSC-IT'
    elif course_string == 'MBA':
         course_string = 'MBA'
    elif course_string == 'MSC-NON-IT':
         course_string = 'MSC'
    elif course_string == 'BSC-NON-IT':
         course_string = 'BSC'
    elif course_string == 'DIPLOMA-IT':
        pass
    elif course_string == 'DIPLOMA':
        pass
    elif course_string == 'MSC-CS-DA':
         course_string = 'MSC-CS, DA'

    # Final check for empty strings or remaining generic terms
    if not course_string or course_string == 'UNKNOWN' or course_string == 'UNSPECIFIED':
        return None

    return course_string

enrolled_df['Education'] = enrolled_df['Course'].apply(standardize_course_name)

print("\nValue counts for 'Education' column after standardization:")
print(enrolled_df['Education'].value_counts())
enrolled_df = enrolled_df.drop(columns=['Course'])
print("Columns in enrolled_df after dropping 'Course':")
print(enrolled_df.columns)


Value counts for 'Education' column after standardization:
Education
BTECH          101
BCA             64
MCA             59
BCOM            31
MSC-IT          27
BSC-IT          21
BSC             19
MBA             15
BA              12
B VOC           11
MSC              7
MTECH            5
GRADUATED        4
B VOC-IT         4
DIPLOMA          4
DIPLOMA-IT       4
MCOM             4
PLUS TWO         2
PG               1
UNSPECIFIED      1
MA               1
Name: count, dtype: int64
Columns in enrolled_df after dropping 'Course':
Index(['Contact Id', 'Contact Owner.id', 'Contact Owner', 'Track Interested',
       'Tag', 'Created By.id', 'Modified By.id', 'Created Time',
       'Modified Time', 'Last Activity Time', 'Contact Name', 'City',
       'Mailing State', 'Mailing Country', 'Program Joined.id',
       'Program Joined', 'Source of lead', 'Pipeline owner.id', 'Semester',
       'Lead Generated on', 'Gender', 'Year of Graduation', 'Experience',
       'Test', 'Followup Email

In [392]:
# Fill null valuesin 'education'
if enrolled_df['Education'].isnull().any():
    # Get the top 12 most frequent non-null values in 'Education'
    top_12_education = enrolled_df['Education'].value_counts().nlargest(12).index.tolist()

    # Identify null indices in the 'Education' column
    null_indices = enrolled_df['Education'].isnull()
    num_nulls = null_indices.sum()

    # Fill null values with random choices from the top 12 values
    if num_nulls > 0 and top_12_education:
        enrolled_df.loc[null_indices, 'Education'] = np.random.choice(top_12_education, size=num_nulls)
        print(f"Filled {num_nulls} null values in 'Education' with random choices from the top 12 values.")
    elif num_nulls == 0:
        print("No null values found in 'Education' column.")
    else:
        print("Warning: The list of top 12 education values is empty, or no nulls to fill.")
else:
    print("No null values found in 'Education' column.")

print("\nValue counts for 'Education' after filling nulls:")
print(enrolled_df['Education'].value_counts())

Filled 877 null values in 'Education' with random choices from the top 12 values.

Value counts for 'Education' after filling nulls:
Education
BTECH          177
BCA            138
MCA            131
BSC            103
MSC-IT         100
BSC-IT          97
BCOM            95
BA              85
MBA             85
MTECH           82
B VOC           81
MSC             75
GRADUATED        4
DIPLOMA          4
B VOC-IT         4
DIPLOMA-IT       4
MCOM             4
PLUS TWO         2
PG               1
UNSPECIFIED      1
MA               1
Name: count, dtype: int64


###### Handling 'Semester'

In [393]:
# Define current year for inference logic
CURRENT_YEAR = 2026

print("Original 'Semester' column value counts (before filling nulls):")
print(enrolled_df['Semester'].value_counts(dropna=False))
print("\n'Semester' column data type:", enrolled_df['Semester'].dtype)

# --- Step 1: Handle 'graduated' candidates (Year of Graduation <= 2025) ---
# The prompt states: "if already graduated that is for year till 2025 candidates are already graduated so make those rows"
# This implies that for candidates with 'Year of Graduation' up to and including 2025,
# if their 'Semester' is null, we should treat them as not actively in a semester.
# Assign 0 to 'Semester' for these cases.

graduated_null_mask = enrolled_df['Semester'].isnull() & (enrolled_df['Year of Graduation'] <= 2025)
enrolled_df.loc[graduated_null_mask, 'Semester'] = 0

print(f"\nFilled {graduated_null_mask.sum()} null 'Semester' values for graduated candidates (Year <= 2025) with 0.")

# --- Step 2: Fill remaining nulls for 'still studying' candidates (Year of Graduation > 2025) ---
# These are candidates whose 'Year of Graduation' is in the future (after 2025, which is the cutoff for 'graduated').
# For these, we will infer a semester based on their 'Education' type,
# by randomly selecting from a typical range of semesters for that education.

remaining_null_mask = enrolled_df['Semester'].isnull()

if remaining_null_mask.sum() > 0:
    # Define typical semester ranges for different education types
    education_to_semesters_range = {
        'BTECH': [1, 2, 3, 4, 5, 6, 7, 8], # 4-year program
        'BE': [1, 2, 3, 4, 5, 6, 7, 8],    # 4-year program
        'BCA': [1, 2, 3, 4, 5, 6],         # 3-year program
        'MCA': [1, 2, 3, 4],               # 2-year program
        'BSC': [1, 2, 3, 4, 5, 6],         # 3-year program
        'BCOM': [1, 2, 3, 4, 5, 6],        # 3-year program
        'MBA': [1, 2, 3, 4],               # 2-year program
        'BSC-IT': [1, 2, 3, 4, 5, 6],      # 3-year program
        'MSC-IT': [1, 2, 3, 4],            # 2-year program
        'B VOC': [1, 2, 3, 4, 5, 6],       # 3-year program
        'MTECH': [1, 2, 3, 4],             # 2-year program
        'BA': [1, 2, 3, 4, 5, 6],          # 3-year program
        'MSC': [1, 2, 3, 4],               # 2-year program
        'DIPLOMA': [1, 2, 3, 4, 5, 6],     # Typically 2 or 3-year program
        'PG': [1, 2, 3, 4],                # Post Graduation, usually 2 years
        'MA': [1, 2, 3, 4],                # 2-year program
        'B VOC-IT': [1, 2, 3, 4, 5, 6],    # 3-year program
        'DIPLOMA-IT': [1, 2, 3, 4, 5, 6],  # 3-year program
        'MCOM': [1, 2, 3, 4]               # 2-year program
    }

    # Default semester range for education types not explicitly listed, or special cases
    default_active_semesters_range = [1, 2, 3, 4] # Common semesters for ongoing study

    for index in enrolled_df[remaining_null_mask].index:
        education = enrolled_df.loc[index, 'Education']

        # Special handling for education types that don't fit into a semester system for active study
        if education in ['PLUS TWO', 'UNSPECIFIED', 'GRADUATED', 0]: # '0' just as a safeguard for any odd values
            enrolled_df.loc[index, 'Semester'] = 0 # Not actively in a degree semester
            continue

        semesters_options = education_to_semesters_range.get(education, default_active_semesters_range)

        # Assign a random semester from the determined range
        if semesters_options:
            enrolled_df.loc[index, 'Semester'] = np.random.choice(semesters_options)
        else:
            # Fallback if somehow semesters_options is empty (shouldn't happen with default)
            enrolled_df.loc[index, 'Semester'] = np.random.choice(default_active_semesters_range)

    print(f"Filled {remaining_null_mask.sum()} remaining null 'Semester' values based on Education.")

else:
    print("No remaining null values found in 'Semester' column to fill after initial graduated fill.")

# --- Step 3: Convert 'Semester' to integer type ---
# Use 'Int64' to allow for NaN values if any remain (though ideally all should be filled)
# and also correctly represent integer semesters.
enrolled_df['Semester'] = enrolled_df['Semester'].astype('Int64')

print("\nFinal 'Semester' column value counts after filling nulls and type conversion:")
print(enrolled_df['Semester'].value_counts(dropna=False))
print("\nFinal 'Semester' column data type:", enrolled_df['Semester'].dtype)

Original 'Semester' column value counts (before filling nulls):
Semester
NaN    1232
6.0      20
4.0      17
2.0       2
8.0       2
3.0       1
Name: count, dtype: int64

'Semester' column data type: float64

Filled 1088 null 'Semester' values for graduated candidates (Year <= 2025) with 0.
Filled 144 remaining null 'Semester' values based on Education.

Final 'Semester' column value counts after filling nulls and type conversion:
Semester
0    1090
2      43
4      42
3      29
1      26
6      23
5      11
7       5
8       5
Name: count, dtype: Int64

Final 'Semester' column data type: Int64


###### Handling 'program joined'

In [394]:
# Unique values in 'program joined'
print(enrolled_df['Program Joined'].unique())

[nan 'DA MAY 2024' 'DA Enrollee September 2024' 'DS Feb 2024'
 'DA March 2024' 'DS March 2024' 'DA JUNE' 'DS JUNE' 'DS MAY 2024'
 'DA JUNE 2024' 'DS November 2024' 'DS AUGUST' 'DA OCTOBER 2024'
 'DS OCTOBER 2024' 'DS Enrollee September 2024' 'DA Online Sep24'
 'DA November 2024' 'Aug DA 2025' 'DA January 2025' 'DS December 2024'
 'July DA Advanced Programme 2025' 'DA DECEMBER 2024' 'DS FEBRUARY 2025'
 'Nov DA 2025' 'Oct DS 2025' 'DS January 25' 'April DS 2025' 'May DS 25'
 'April DA 2025' 'DS March 2025' 'June DS 2025' 'Aug DS 2025' 'May DA 25'
 'June DA 2025' 'June MERN 2025' 'Sep DS 2025' 'July DA 2025'
 'July DS 2025' 'Aug MERN 2025' 'Feb DS 2026' 'Sep DA 2025' 'Oct DA 2025'
 'Sep DA & DS Advanced programme' 'Nov DS 2025' 'Dec DA 2025'
 'Nov MERN 2025' 'Dec DS 2025' 'Jan DA 2026' 'Jan DS 2026' 'Feb DA 2026'
 'MERN October 2024' 'DA APRIL 2024' 'DA JULY' 'DS JULY' 'DA March 2025'
 'MERN November 24' 'MERN December2024' 'MERN STACK FEB 2025'
 'DA Feb 2025' 'May MERN 25' 'May PEP 25' '

In [395]:
# Standardize the values in 'program joined'
def standardize_program_joined_text(program_string):
    if pd.isna(program_string):
        return None

    program_string = str(program_string).strip()

    # 1. Standardize month names (full to short form, and ensure proper case)
    month_mapping = {
        'January': 'Jan', 'February': 'Feb', 'March': 'Mar', 'April': 'Apr',
        'May': 'May', 'June': 'Jun', 'July': 'Jul', 'August': 'Aug',
        'September': 'Sep', 'October': 'Oct', 'November': 'Nov', 'December': 'Dec'
    }
    for full, short in month_mapping.items():
        program_string = re.sub(r'\b' + full + r'\b', short, program_string, flags=re.IGNORECASE)

    # Ensure month abbreviations are capitalized (e.g., "jan" -> "Jan")
    program_string = re.sub(r'\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b', lambda x: x.group(0).capitalize(), program_string, flags=0)

    # 2. Standardize common program keywords/abbreviations (case-insensitive search, consistent replacement)
    program_string = re.sub(r'\bprogramme\b', 'Program', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\b(enroll|enrollee)\b', 'Enrollee', program_string, flags=re.IGNORECASE)

    # Specific course abbreviations - these should be uppercase, or full name as requested
    program_string = re.sub(r'\bmern stack\b', 'MERN', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bmern\b', 'MERN', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bdata analytics\b', 'DA', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bdata analytic\b', 'DA', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bdata science\b', 'DS', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bdata sci\b', 'DS', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bfull stack\b', 'Full Stack', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bpy\b', 'Python', program_string, flags=re.IGNORECASE)

    # Add specific full program names
    program_string = re.sub(r'\bRounded Data Science & GenAI Professional\b', 'Rounded DS & GenAI Professional', program_string, flags=re.IGNORECASE)
    program_string = re.sub(r'\bGen AI & Prompt Engineering\b', 'Gen AI & Prompt Engineering', program_string, flags=re.IGNORECASE)

    # Standardize 'One Month Program' variations
    program_string = re.sub(r'\b(One Month Programme|One Month Program)\b', 'One Month Program', program_string, flags=re.IGNORECASE)

    # Clean up extra spaces and ensure consistent casing for whole string
    program_string = ' '.join(program_string.split()).title()

    # Fix abbreviations that might have been title-cased incorrectly
    program_string = program_string.replace('Da', 'DA').replace('Ds', 'DS').replace('Mern', 'MERN').replace('Py', 'Python')
    program_string = program_string.replace('Program', 'Program').replace('Enrollee', 'Enrollee') # Maintain specific capitalizations
    # Explicitly ensure 'Full Stack' remains capitalized after title() if it was already correct
    program_string = program_string.replace('Full Stack', 'Full Stack')
    program_string = program_string.replace('Rounded Ds & Genai Professional', 'Rounded DS & GenAI Professional')
    program_string = program_string.replace('Gen Ai & Prompt Engineering', 'Gen AI & Prompt Engineering')

    return program_string

# Apply this initial standardization to the 'Program Joined' column
enrolled_df['Program Joined'] = enrolled_df['Program Joined'].apply(standardize_program_joined_text)

print("Value counts for 'Program Joined' after initial standardization:")
print(enrolled_df['Program Joined'].value_counts())

Value counts for 'Program Joined' after initial standardization:
Program Joined
Jan DA 2026                     48
Aug DS 2025                     40
Aug DA 2025                     39
Nov DA 2025                     38
Jan DS 2026                     34
                                ..
Oct One Month Program 2025       1
Jan MERN 2026                    1
MERN Jan25                       1
Nov One Month Program 2025       1
Oct DA & DS Advanced Program     1
Name: count, Length: 84, dtype: int64


In [396]:
# Create new column 'batch assigned' from program joined
def extract_batch_assigned(program_joined):
    if pd.isna(program_joined):
        return None

    program_joined = str(program_joined)

    # Try to find a 4-digit year first
    year_match = re.search(r'\b(20\d{2})\b', program_joined)
    year = year_match.group(1) if year_match else None

    # Extract month abbreviation
    month_match = re.search(r'\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\b', program_joined)
    month = month_match.group(1) if month_match else None

    if month and year:
        return f"{month} {year}"
    elif month and not year:
        # If only month is found, return only month
        return month
    return None

# Apply the function to create the 'Batch Assigned' column
enrolled_df['Batch Assigned'] = enrolled_df['Program Joined'].apply(extract_batch_assigned)

print("\nValue counts for 'Batch Assigned' column:")
print(enrolled_df['Batch Assigned'].value_counts().head(10))


Value counts for 'Batch Assigned' column:
Batch Assigned
Aug 2025    87
Jan 2026    84
Oct 2024    58
Jul 2025    58
Nov 2025    58
Sep 2024    56
Jun 2025    56
Oct 2025    55
Dec 2024    55
Apr 2025    51
Name: count, dtype: int64


###### Handling 'Batch Assigned'

In [397]:
# --- New: Function to clean duplicate years in 'Batch Assigned' ---
def clean_duplicate_years(batch_string):
    if pd.isna(batch_string):
        return None

    s = str(batch_string).strip()

    # Pattern to find a month abbreviation and a 4-digit year, possibly with extra years or text
    # We want to capture only the first valid month-year combination
    match = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+(20\d{2})', s, re.IGNORECASE)

    if match:
        month = match.group(1).capitalize() # Ensure month is capitalized
        year = match.group(2)
        return f"{month} {year}"
    else:
        # If no month-year combination, check for month alone
        month_match = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)', s, re.IGNORECASE)
        if month_match:
            return month_match.group(1).capitalize()

    return None # Return None if no valid month or month-year is found

# Apply the cleaning function first to remove duplicate years
print("Cleaning 'Batch Assigned' entries with duplicate years...")
enrolled_df['Batch Assigned'] = enrolled_df['Batch Assigned'].apply(clean_duplicate_years)
print("Finished cleaning duplicate years.")

if enrolled_df['Batch Assigned'].isnull().any():
    # Get unique non-null values from the 'Batch Assigned' column
    existing_batches = enrolled_df['Batch Assigned'].dropna().unique().tolist()

    # Identify null indices in the 'Batch Assigned' column
    null_indices = enrolled_df['Batch Assigned'].isnull()
    num_nulls = null_indices.sum()

    # Fill null values with random choices from the existing batches
    if num_nulls > 0 and existing_batches:
        enrolled_df.loc[null_indices, 'Batch Assigned'] = np.random.choice(existing_batches, size=num_nulls)
        print(f"Filled {num_nulls} null values in 'Batch Assigned' with random choices from existing batches.")
    elif num_nulls == 0:
        print("No null values found in 'Batch Assigned' column.")
    else:
        print("Warning: The list of existing batches is empty, or no nulls to fill.")
else:
    print("No null values found in 'Batch Assigned' column.")

# --- Logic to add random year to month-only entries ---
print("\nStandardizing 'Batch Assigned' entries that are month-only...")

# Pre-compile the regex pattern for efficiency and robustness
# This pattern checks for a 4-digit year starting with '20' that is treated as a whole word.
YEAR_PATTERN_DETECT = re.compile(r'\b20\d{2}\b') # Raw string for word boundary and year

# Function to check if a string contains a year using the pre-compiled pattern
def contains_year(batch_string):
    return bool(YEAR_PATTERN_DETECT.search(str(batch_string)))

# Define a list of possible years for random assignment
possible_years = [2024, 2025, 2026]

# Iterate through 'Batch Assigned' and update month-only entries
modified_count = 0
for idx, batch_value in enrolled_df['Batch Assigned'].items():
    if pd.isna(batch_value):
        continue

    current_val_str = str(batch_value).strip() # Strip spaces before checking

    # Check if the batch value already contains a year
    if not contains_year(current_val_str):
        # It's a month-only string, append a random year
        random_year = np.random.choice(possible_years)
        enrolled_df.loc[idx, 'Batch Assigned'] = f"{current_val_str} {random_year}"
        modified_count += 1

print(f"Modified {modified_count} month-only entries by adding a random year.")
print("Standardization complete.")
print("\nValue counts for 'Batch Assigned' after filling nulls and further standardization:")
print(enrolled_df['Batch Assigned'].value_counts())

Cleaning 'Batch Assigned' entries with duplicate years...
Finished cleaning duplicate years.
Filled 255 null values in 'Batch Assigned' with random choices from existing batches.

Standardizing 'Batch Assigned' entries that are month-only...
Modified 175 month-only entries by adding a random year.
Standardization complete.

Value counts for 'Batch Assigned' after filling nulls and further standardization:
Batch Assigned
Jan 2026    102
Aug 2025    101
Jun 2025     79
Oct 2024     69
Oct 2025     67
Nov 2025     65
Sep 2024     65
Dec 2024     65
Jul 2025     64
Apr 2025     61
Dec 2025     55
Sep 2025     55
May 2024     52
Nov 2024     51
Feb 2026     51
Jan 2025     38
Feb 2025     32
Jan 2024     31
Mar 2025     25
Jun 2024     22
May 2025     22
Apr 2024     21
May 2026     19
Feb 2024     16
Mar 2024     10
Jun 2026      9
Jul 2024      6
Jul 2026      6
Sep 2026      5
Nov 2026      4
Aug 2026      3
Aug 2024      2
Oct 2026      1
Name: count, dtype: int64


In [398]:
# Create new column 'course' from program joined
def extract_course_name(program_joined):
    if pd.isna(program_joined):
        return None

    program_joined_upper = str(program_joined).upper()

    # Most specific programs first
    if 'ROUNDED DS & GENAI PROFESSIONAL' in program_joined_upper:
        return 'Rounded DS & GenAI Professional'
    elif 'GEN AI & PROMPT ENG' in program_joined_upper:
        return 'Gen AI & Prompt Engineering'
    elif 'DA & DS ADVANCED PROGRAM' in program_joined_upper:
        return 'DA & DS Advanced Program'
    elif 'ONE MONTH PROGRAM' in program_joined_upper or 'ONE MONTH' in program_joined_upper:
        return 'One Month Program'

    # Specific abbreviations
    elif 'MERN' in program_joined_upper:
        return 'MERN Stack'
    elif 'FULL STACK' in program_joined_upper or 'FS' in program_joined_upper:
        return 'Full Stack'
    elif 'DS' in program_joined_upper:
        return 'Data Science'
    elif 'DA' in program_joined_upper:
        return 'Data Analytics'
    elif 'PYTHON' in program_joined_upper:
        return 'Python'
    elif 'PEP' in program_joined_upper:
        return 'PEP Program'

    # Fallback for anything not caught
    return 'Other Course'

# Apply the function to create the 'Course' column
enrolled_df['Course'] = enrolled_df['Program Joined'].apply(extract_course_name)

print("\nValue counts for 'Course' column after refined extraction:")
print(enrolled_df['Course'].value_counts())
print("\nUnique values mapped to 'Other Course' (if any):")
print(enrolled_df[enrolled_df['Course'] == 'Other Course']['Program Joined'].unique())


Value counts for 'Course' column after refined extraction:
Course
Data Analytics                     540
Data Science                       426
MERN Stack                          52
One Month Program                    8
Rounded DS & GenAI Professional      3
PEP Program                          2
DA & DS Advanced Program             2
Python                               1
Gen AI & Prompt Engineering          1
Name: count, dtype: int64

Unique values mapped to 'Other Course' (if any):
[]


###### Handling 'Experience'

In [399]:
# Ensure CURRENT_YEAR is defined if not already from previous cells
try:
    CURRENT_YEAR
except NameError:
    CURRENT_YEAR = 2026 # Default to 2026 if not set elsewhere

print("Original 'Experience' column null count:")
print(enrolled_df['Experience'].isnull().sum())

# --- Step 1: Handle 'not yet graduated' candidates ---
# Candidates whose 'Year of Graduation' is greater than CURRENT_YEAR are considered not yet graduated
# They should have 0 experience.
not_graduated_null_experience_mask = enrolled_df['Experience'].isnull() & (enrolled_df['Year of Graduation'] > CURRENT_YEAR)
enrolled_df.loc[not_graduated_null_experience_mask, 'Experience'] = 0

print(f"Filled {not_graduated_null_experience_mask.sum()} null 'Experience' values for 'not yet graduated' candidates with 0.")

# --- Step 2: Handle remaining nulls for 'graduated' candidates ---
# For those who have graduated (Year of Graduation <= CURRENT_YEAR) but have null Experience,
# fill with a random number of years between 0 and (CURRENT_YEAR - Year of Graduation).
remaining_null_experience_mask = enrolled_df['Experience'].isnull()
num_remaining_nulls = remaining_null_experience_mask.sum()

if num_remaining_nulls > 0:
    # Iterate through the rows with remaining null 'Experience'
    for idx in enrolled_df[remaining_null_experience_mask].index:
        year_grad = enrolled_df.loc[idx, 'Year of Graduation']

        # Calculate the maximum possible experience, ensuring it's not negative
        max_possible_experience = max(0, CURRENT_YEAR - year_grad)

        # Fill with a random integer from 0 up to max_possible_experience (inclusive)
        enrolled_df.loc[idx, 'Experience'] = np.random.randint(0, max_possible_experience + 1)

    print(f"Filled {num_remaining_nulls} remaining null 'Experience' values with random years (0 to CURRENT_YEAR - Year of Graduation).")
else:
    print("No remaining null values found in 'Experience' to fill.")

# --- Step 3: Convert 'Experience' to integer type ---
# Use 'Int64' to allow for NaN values if any remain (though ideally all should be filled)
# and also correctly represent integer experience.
enrolled_df['Experience'] = enrolled_df['Experience'].astype('Int64')

print("\nFinal 'Experience' column null count:")
print(enrolled_df['Experience'].isnull().sum())

print("\nValue counts for 'Experience' after filling and type conversion (top 10):")
print(enrolled_df['Experience'].value_counts().head(10))
print(f"\nFinal 'Experience' column data type: {enrolled_df['Experience'].dtype}")

Original 'Experience' column null count:
1260
Filled 76 null 'Experience' values for 'not yet graduated' candidates with 0.
Filled 1184 remaining null 'Experience' values with random years (0 to CURRENT_YEAR - Year of Graduation).

Final 'Experience' column null count:
0

Value counts for 'Experience' after filling and type conversion (top 10):
Experience
0    398
1    242
2    181
3    121
4     97
5     75
6     49
7     37
9     23
8     22
Name: count, dtype: Int64

Final 'Experience' column data type: Int64


In [400]:
## Create the 'role' column based on conditional logic
def assign_role(row):
    if row['Experience'] > 0:
        return 'Professional'
    elif row['Semester'] > 0:
        return 'Student'
    elif row['Year of Graduation'] > 0 and row['Experience'] == 0:
        # All 'Year of Graduation' values are now integers > 0, so this simplifies to Experience == 0
        return 'Idle or Career Gap'
    return 'Unknown' # Fallback for unexpected cases, though ideally all should be covered

enrolled_df['role'] = enrolled_df.apply(assign_role, axis=1)
print(enrolled_df['role'].value_counts())

role
Professional          876
Idle or Career Gap    234
Student               164
Name: count, dtype: int64


In [401]:
## Create the 'background' column based on course
def assign_background(course):
    if pd.isna(course) or course == 'NOT MENTIONED' or course == 'UNSPECIFIED':
        return 'UNKNOWN'

    # Define lists of common tech and non-tech courses
    tech_keywords = [
        'BTECH', 'BE', 'MTECH', 'BCA', 'MCA', 'B VOC-IT', 'BSC-CS', 'MSC-CS', 'CSE', 'CS', 'IT', 'DA', 'DS', 'BDA',
        'MSCIT', 'BSCIT', 'MTECHIT', 'MSC-CS-DA', 'BTECH-IT', 'BSC-IT', 'DIPLOMA-IT'
    ]

    non_tech_keywords = [
        'BCOM', 'MCOM', 'BA', 'MBA', 'MA', 'B VOC', 'BSC', 'MSC', 'ENG LIT', 'PLUS TWO', 'DIPLOMA', 'PG',
        'BSC-NON-IT', 'DIPLOMA-NON-IT', 'GRADUATED', 'OTHERS'
    ]

    # Convert course to uppercase for case-insensitive matching
    course_upper = str(course).upper()

    for keyword in tech_keywords:
        if keyword in course_upper:
            return 'Tech'

    for keyword in non_tech_keywords:
        if keyword in course_upper:
            return 'Non-Tech'

    # If none of the above match, try to infer based on broader terms or default
    if 'TECH' in course_upper or 'SCIENCE' in course_upper or 'ENGINEERING' in course_upper:
        return 'Tech'
    elif 'ARTS' in course_upper or 'COMMERCE' in course_upper or 'HUMANITIES' in course_upper:
        return 'Non-Tech'

    return 'UNKNOWN'

enrolled_df['background'] = enrolled_df['Education'].apply(assign_background)
print(enrolled_df['background'].value_counts())


background
Tech        733
Non-Tech    540
UNKNOWN       1
Name: count, dtype: int64


In [402]:
# Define the possible values for the 'Stream' column
stream_values = ['Project', 'Training']

# Fill the new 'Stream' column with random choices from the defined values
enrolled_df['Stream'] = np.random.choice(stream_values, size=len(enrolled_df))

print("Value counts for the new 'Stream' column:")
print(enrolled_df['Stream'].value_counts())

Value counts for the new 'Stream' column:
Stream
Project     655
Training    619
Name: count, dtype: int64


In [403]:
# Define the possible values for the 'Induction session' column
induction_session_values = ['Attended', 'Not Attended']

# Define the probabilities for each value (e.g., 70% 'Attended', 30% 'Not Attended')
probabilities = [0.7, 0.3]

# Fill the new 'Induction session' column with random choices based on probabilities
enrolled_df['Induction session'] = np.random.choice(induction_session_values, size=len(enrolled_df), p=probabilities)

print("Value counts for the new 'Induction session' column:")
print(enrolled_df['Induction session'].value_counts())

Value counts for the new 'Induction session' column:
Induction session
Attended        887
Not Attended    387
Name: count, dtype: int64


In [404]:
# Create new column 'Feedback'
feedback_values = ['Positive', 'Neutral', 'Negative']

# Define a function to assign feedback based on induction session attendance
def assign_feedback(row):
    if row['Induction session'] == 'Not Attended':
        return 'Not Attended'
    else:
        # Randomly choose from Positive, Neutral, Negative with specified probabilities
        return np.random.choice(feedback_values, p=[0.5, 0.3, 0.2]) # More Positive, then Neutral, then Negative

# Apply the function to create the new 'Feedback' column
enrolled_df['Feedback'] = enrolled_df.apply(assign_feedback, axis=1)

print("Value counts for the new 'Feedback' column:")
print(enrolled_df['Feedback'].value_counts())

Value counts for the new 'Feedback' column:
Feedback
Positive        449
Not Attended    387
Neutral         260
Negative        178
Name: count, dtype: int64


In [405]:
# Define a function to assign Total_Amount based on Course and Stream
def assign_total_amount(row):
    course = row['Course']
    stream = row['Stream']

    if stream == 'Project':
        # For 'Project' stream, choose from the specified round numbers
        return random.choice([10000, 20000, 30000, 40000, 50000, 60000])
    elif stream == 'Training':
        # For 'Training' stream, assign different specific values for each course
        if course == 'Data Analytics':
            return 100000
        elif course == 'Data Science':
            return 120000
        elif course == 'MERN Stack':
            return 90000
        elif course == 'One Month Program':
            return 25000 # Lower for one month program
        elif course == 'Rounded DS & GenAI Professional':
            return 150000 # Higher for professional program
        elif course == 'PEP Program':
            return 35000
        elif course == 'DA & DS Advanced Program':
            return 130000
        elif course == 'Python':
            return 70000
        elif course == 'Gen AI & Prompt Engineering':
            return 110000
        else:
            # Fallback for any other course under 'Training' if not explicitly defined
            return 50000
    return 0 # Default if stream is neither Project nor Training (should not occur with current data)

# Apply the function to create the 'Total_Amount' column
enrolled_df['Total_Amount'] = enrolled_df.apply(assign_total_amount, axis=1)

print("Value counts for 'Total_Amount' by 'Stream' (head):")
print(enrolled_df.groupby('Stream')['Total_Amount'].value_counts().head(20))

Value counts for 'Total_Amount' by 'Stream' (head):
Stream    Total_Amount
Project   10000           124
          40000           121
          30000           113
          20000           110
          60000            94
          50000            93
Training  100000          264
          120000          208
          50000           121
          90000            17
          25000             3
          130000            2
          150000            2
          35000             1
          110000            1
Name: count, dtype: int64


In [406]:
# Define the paid amount
round_figures_options = [10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000]

def assign_paid_amount(row):
    invoice = row['Invoice']
    total_amount = row['Total_Amount']

    # Rule 1: If invoice is 'No', paid amount should be 0
    if invoice == 'No':
        return 0

    # Invoice is 'Yes', proceed with other rules
    valid_paid_options = []

    # Rule 2: Include 2000 as a possible option if it's less than total_amount
    if 2000 < total_amount:
        valid_paid_options.append(2000)

    # Rule 3: Include round figures that are strictly less than total_amount
    for fig in round_figures_options:
        if fig < total_amount:
            valid_paid_options.append(fig)

    # If there are no valid options (e.g., total_amount is very low, say 1000),
    # then the paid amount defaults to 0.
    if not valid_paid_options:
        return 0
    else:
        # Randomly choose one of the valid options
        return random.choice(valid_paid_options)

# Apply the function to create the 'Paid_amount' column
enrolled_df['Paid_amount'] = enrolled_df.apply(assign_paid_amount, axis=1)

print("Value counts for 'Paid_amount':")
print(enrolled_df['Paid_amount'].value_counts())

# --- Verification ---
print("\n--- Verification ---")

# 1. Check for Invoice 'No': Paid_amount must be 0
invoice_no_paid_amount_check = enrolled_df[enrolled_df['Invoice'] == 'No']['Paid_amount'].unique()
print(f"Unique Paid_amount values for 'Invoice' = 'No': {invoice_no_paid_amount_check} (should be [0])")
if not (len(invoice_no_paid_amount_check) == 1 and invoice_no_paid_amount_check[0] == 0):
    print("WARNING: Paid_amount for 'Invoice' = 'No' is not exclusively 0.")

# 2. Check Paid_amount < Total_Amount for all entries where Paid_amount > 0
invalid_paid_amount_count = enrolled_df[
    (enrolled_df['Paid_amount'] > 0) &
    (enrolled_df['Paid_amount'] >= enrolled_df['Total_Amount'])
].shape[0]
print(f"Number of cases where Paid_amount >= Total_Amount (and Paid_amount > 0): {invalid_paid_amount_count} (should be 0)")
if invalid_paid_amount_count > 0:
    print("WARNING: Found cases where Paid_amount is not strictly less than Total_Amount.")

# 3. Check if any Paid_amount values for Invoice 'Yes' are within the specified valid set (2000, round figures, or 0)
invoice_yes_df = enrolled_df[enrolled_df['Invoice'] == 'Yes']
valid_paid_set = set([0, 2000] + round_figures_options)

invoice_yes_invalid_paid_count = invoice_yes_df[~invoice_yes_df['Paid_amount'].isin(valid_paid_set)].shape[0]
print(f"Number of 'Invoice' = 'Yes' entries with Paid_amount not in {{2000, round_figures, 0}}: {invoice_yes_invalid_paid_count} (should be 0)")
if invoice_yes_invalid_paid_count > 0:
    print("WARNING: Some 'Paid_amount' values for 'Invoice' = 'Yes' are not in the specified valid set.")


Value counts for 'Paid_amount':
Paid_amount
2000      289
0         190
10000     188
20000     145
30000     118
40000      71
50000      60
70000      50
90000      48
80000      47
60000      41
100000     27
Name: count, dtype: int64

--- Verification ---
Unique Paid_amount values for 'Invoice' = 'No': [0] (should be [0])
Number of cases where Paid_amount >= Total_Amount (and Paid_amount > 0): 0 (should be 0)
Number of 'Invoice' = 'Yes' entries with Paid_amount not in {2000, round_figures, 0}: 0 (should be 0)


In [407]:
# Define 'payment mode'
def assign_payment_mode(row):
    if row['Paid_amount'] == 0:
        return 'Not Paid'
    else:
        return random.choice(['Cash', 'UPI', 'Check'])

# Apply the function to create the 'Payment_mode' column
enrolled_df['Payment_mode'] = enrolled_df.apply(assign_payment_mode, axis=1)

print("Value counts for 'Payment_mode':")
print(enrolled_df['Payment_mode'].value_counts())

Value counts for 'Payment_mode':
Payment_mode
Cash        393
UPI         365
Check       326
Not Paid    190
Name: count, dtype: int64


In [408]:
#Define 'payment date'
# Convert 'Last Activity Time' to datetime, handling potential errors and nulls
enrolled_df['Last Activity Time'] = pd.to_datetime(enrolled_df['Last Activity Time'], format='%d-%m-%Y %H:%M', errors='coerce')

def generate_payment_date(row):
    last_activity_dt = row['Last Activity Time']

    # If 'Last Activity Time' is NaT, return NaT for 'Payment_Date'
    if pd.isna(last_activity_dt):
        return pd.NaT

    # Try to generate a date in the same month as last_activity_dt
    if last_activity_dt.day >= 10:
        # If last_activity_dt's day is 10 or greater, we can pick a day in the same month
        # ensuring it's not after last_activity_dt and is between 10-20.
        possible_days = list(range(10, min(20, last_activity_dt.day) + 1))
        if possible_days:
            chosen_day = random.choice(possible_days)
            return last_activity_dt.replace(day=chosen_day, hour=0, minute=0, second=0, microsecond=0)

    # If last_activity_dt's day is less than 10, or no suitable day found in current month (unlikely for 10-20 if day >= 10)
    # we must go to the previous month.
    prev_month_dt = last_activity_dt - relativedelta(months=1)

    try:
        # Get the last day of the previous month to determine the valid range for `random.randint`
        max_day_in_prev_month = (prev_month_dt.replace(day=1) + relativedelta(months=1, days=-1)).day
        possible_days = list(range(10, min(20, max_day_in_prev_month) + 1))
        if possible_days:
            chosen_day = random.choice(possible_days)
            return prev_month_dt.replace(day=chosen_day, hour=0, minute=0, second=0, microsecond=0)
    except ValueError:
        # Fallback if previous month logic has issues (e.g., prev_month_dt is invalid), should be rare.
        pass

    # Final fallback: if no valid date could be generated adhering to the 10-20 day rule,
    # return the last_activity_dt itself (normalized to start of day) to satisfy the <= constraint.
    return last_activity_dt.replace(hour=0, minute=0, second=0, microsecond=0)

# Apply the function to create the 'Payment_Date' column
enrolled_df['Payment_Date'] = enrolled_df.apply(generate_payment_date, axis=1)

# Format the 'Payment_Date' column to 'DD-MM-YYYY' string
enrolled_df['Payment_Date'] = enrolled_df['Payment_Date'].dt.strftime('%d-%m-%Y')

print("Value counts for 'Payment_Date' (first 10):")
print(enrolled_df['Payment_Date'].value_counts().head(10))

print("\nVerifying Payment_Date <= Last Activity Time for a few rows:")
print(enrolled_df[['Last Activity Time', 'Payment_Date']].head())

# Additional verification: Check if any Payment_Date is after Last Activity Time
# Convert 'Payment_Date' back to datetime for accurate comparison
enrolled_df['Payment_Date_dt_verify'] = pd.to_datetime(enrolled_df['Payment_Date'], format='%d-%m-%Y', errors='coerce')
# Normalize 'Last Activity Time' to remove time component for date-only comparison
enrolled_df['Last Activity Time_date_only_verify'] = enrolled_df['Last Activity Time'].dt.normalize()

invalid_payment_dates = enrolled_df[enrolled_df['Payment_Date_dt_verify'] > enrolled_df['Last Activity Time_date_only_verify']]
print(f"\nNumber of cases where Payment_Date > Last Activity Time: {len(invalid_payment_dates)}")
if not invalid_payment_dates.empty:
    print("Cases violating Payment_Date <= Last Activity Time constraint:")
    print(invalid_payment_dates[['Last Activity Time', 'Payment_Date', 'Payment_Date_dt_verify', 'Last Activity Time_date_only_verify']].head())

# Clean up temporary verification columns
enrolled_df = enrolled_df.drop(columns=['Payment_Date_dt_verify', 'Last Activity Time_date_only_verify'])

Value counts for 'Payment_Date' (first 10):
Payment_Date
11-05-2025    32
10-12-2025    32
16-05-2025    29
10-02-2026    28
19-05-2025    28
10-05-2025    27
12-05-2025    26
18-05-2025    24
17-05-2025    24
13-05-2025    23
Name: count, dtype: int64

Verifying Payment_Date <= Last Activity Time for a few rows:
   Last Activity Time Payment_Date
0 2024-09-25 19:25:00   15-09-2024
1 2024-09-25 19:26:00   11-09-2024
2 2024-09-25 19:28:00   19-09-2024
3 2025-05-19 12:49:00   10-05-2025
4 2024-11-08 14:34:00   19-10-2024

Number of cases where Payment_Date > Last Activity Time: 0


In [409]:
# Fill null values in 'Program joined'
print(f"Null values in 'Program Joined' before filling: {enrolled_df['Program Joined'].isnull().sum()}")

enrolled_df['Program Joined'] = enrolled_df['Program Joined'].fillna('Not Joined')

print(f"Null values in 'Program Joined' after filling: {enrolled_df['Program Joined'].isnull().sum()}")
print("\nValue counts for 'Program Joined' after filling:")
print(enrolled_df['Program Joined'].value_counts().head(10))

Null values in 'Program Joined' before filling: 239
Null values in 'Program Joined' after filling: 0

Value counts for 'Program Joined' after filling:
Program Joined
Not Joined              239
Jan DA 2026              48
Aug DS 2025              40
Aug DA 2025              39
Nov DA 2025              38
Jan DS 2026              34
DA Oct 2024              34
Oct DA 2025              33
Feb DA 2026              33
DA Enrollee Sep 2024     31
Name: count, dtype: int64


###### Handling notes dataset

In [413]:
# Remove 'Note Title' in the notes dataset
notes = notes.drop(columns = ['Note Title'])

# Fill null values for 'Note Content' in the notes dataset
notes['Note Content'] = notes['Note Content'].fillna('Not available')

###### Handling 'notes content'

In [414]:
# Cleaning 'notes content' values by tokenising
# Download necessary NLTK data (run once)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
# Remove 'not' from stopwords to retain negation meaning
stop_words.discard('not')

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text) or text == 'Not available':
        return []
    text = str(text).lower() # Lowercase
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    words = text.split() # Tokenization
    words = [word for word in words if word not in stop_words] # Remove stopwords
    words = [lemmatizer.lemmatize(word) for word in words] # Lemmatization
    return words

notes['cleaned_content'] = notes['Note Content'].apply(preprocess_text)

print("First 5 rows of notes with cleaned content:")
print(notes[['Note Content', 'cleaned_content']].head())

First 5 rows of notes with cleaned content:
                                   Note Content  \
0                       SIJINA\nANOTHER CALLING   
1  SONU\nDATA ANALYTICS ALREADY INTERNSHIP DONE   
2                                   JOINED WORK   
3                           currently doing job   
4                         SHILKA\nNOT CONNECTED   

                                     cleaned_content  
0                         [sijina, another, calling]  
1  [sonu, data, analytics, already, internship, d...  
2                                     [joined, work]  
3                                   [currently, job]  
4                           [shilka, not, connected]  


In [415]:
# Defining cleaned sentences context
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# frequent word context analsis
def get_sentences(text):
    if pd.isna(text) or text == 'Not available':
        return []

    text_str = str(text)

    # Replace newline characters with a period and space for better sentence tokenization
    # Handle both '\n' (literal) and '\n' (escaped) for robustness
    processed_text = re.sub(r'\\n+', '. ', text_str) # For escaped newlines
    processed_text = re.sub(r'\n+', '. ', processed_text)  # For actual newline characters
    # Clean up multiple periods that might result from successive newlines
    processed_text = re.sub(r'\s*\.\s*\.\s*', '. ', processed_text)
    # Ensure there's a space after periods for proper tokenization
    processed_text = re.sub(r'\.(?!\s)', '. ', processed_text)

    # Tokenize the processed text into sentences
    raw_sentences = sent_tokenize(processed_text)

    cleaned_sentences = []
    for sentence in raw_sentences:
        # Clean and strip the sentence for heuristic checks
        # Temporarily remove punctuation for a more accurate 'all-caps single word' check
        cleaned_s_for_check = re.sub(r'[^a-zA-Z0-9\s]', '', sentence).strip()

        # Heuristic to filter out potential 'names' or very short non-informative phrases
        # If a sentence becomes just one word after cleaning and is all uppercase (and not just a single letter), it's likely a name.
        words = cleaned_s_for_check.split()
        if len(words) == 1 and words[0].isupper() and len(words[0]) > 1:
            continue # Filter out single, all-caps words that are likely names (e.g., 'SIJINA')

        # If the original sentence (after stripping) is meaningful, add it
        if sentence.strip():
            cleaned_sentences.append(sentence.strip())

    return cleaned_sentences

notes['sentences'] = notes['Note Content'].apply(get_sentences)

print("First 5 rows of notes with tokenized sentences:")
print(notes[['Note Content', 'sentences']].head())

First 5 rows of notes with tokenized sentences:
                                   Note Content  \
0                       SIJINA\nANOTHER CALLING   
1  SONU\nDATA ANALYTICS ALREADY INTERNSHIP DONE   
2                                   JOINED WORK   
3                           currently doing job   
4                         SHILKA\nNOT CONNECTED   

                                  sentences  
0                         [ANOTHER CALLING]  
1  [DATA ANALYTICS ALREADY INTERNSHIP DONE]  
2                             [JOINED WORK]  
3                     [currently doing job]  
4                           [NOT CONNECTED]  


In [416]:
# Extracting reason from 'notes content'
def infer_status_and_reason_from_notes(sentences_list):
    full_note_text = ' '.join(sentences_list).lower()

    # Keywords for 'Joined' status - these should take highest precedence
    joined_keywords = [
        'enrolled', 'paid fees', 'paid the fees', 'will join today', 'joined', 'registered', 'starts program',
        'course started', 'confirmed admission', 'done payment', 'admission confirmed',
        'assessment', 'assessment attended', 'joined today', 'start classes', 'class started'
    ]

    # --- Not Joined Reasons Keywords ---
    competitor_names_list = ['luminar', 'avodha', 'smec', 'liuminar', 'techminds', 'soften', 'techolas', 'lumimar', 'other institute', 'lum', 'excelr', 'freshers job', 'xlr']
    completed_phrases_list = ['already done', 'already completed', 'already did', 'aloready completed', 'doing', 'percuing', 'done with internship'] # Typos considered

    # Keywords for "Join Later"
    join_later_keywords = ['join later', 'will join', 'joining next month', 'joining in', 'joining soon']
    month_names = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

    # New explicit phrases for "Details Shared/Collected (Candidate)"
    candidate_details_phrases = [
        'candidate details shared', 'student details shared', 'profile details shared',
        'contact details shared', 'information shared about candidate',
        'collected candidate details', 'collected student details', 'collected profile details',
        'collected contact details', 'collected information about candidate',
        'details shared with candidate', 'details collected from candidate'
    ]

    # Check for 'Joined' status first
    for keyword in joined_keywords:
        if keyword in full_note_text:
            return 'Joined', 'N/A'

    # --- Join Later Status Logic (moved to higher precedence) ---
    if any(keyword in full_note_text for keyword in join_later_keywords):
        return 'Join Later', 'N/A'
    # Check for 'month name later' pattern
    for month in month_names:
        if re.search(r'\b' + month + r'\s+later\b', full_note_text):
            return 'Join Later', 'N/A'

    # --- Not Joined Reasons Logic ---

    # Priority 1: Specific check for 'Joined Competitor' when combined with 'already done' or 'already completed'
    found_competitor_keyword = any(comp_name in full_note_text for comp_name in competitor_names_list)
    found_completed_phrase = any(comp_phrase in full_note_text for comp_phrase in completed_phrases_list)

    if found_competitor_keyword and found_completed_phrase:
        return 'Not Joined', 'Joined Competitor'

    # Priority 2: Other 'Not Joined' reasons
    not_joined_reasons = {
        'Joined Competitor': [
            'lum', 'luminar', 'avodha', 'excelr', 'smec', 'liuminar', 'techminds', 'soften', 'joined competitor', 'other institute', 'lumimar', 'techolas',
            'freshers job'
        ],
        'Already Working/Internship': [
            'already working', 'already job', 'technopark', 'infopark', 'doing job', 'doing internship', 'working experience', 'placed', 'joined work', 'already internship done', 'currently working',
            'working as a', 'current job', 'previous job', 'passed out', 'got job', 'job offer'
        ],
        'Looking for Job/Internship (Specific Type)': [
            'looking for job', 'job only', 'looking for internship', 'internship only', 'looking for stipended internship',
            'looking for stipend', 'looking for stipended', 'free internship', 'only job', 'only internship', 'stipend internship', 'paid internship'
        ],
        'Not Interested': [
            'not interested', 'no interest', 'not joining', 'not join', 'not intrested for internship', 'not wish to join'
        ],
        'Financial Issue': [
            'fees', 'amount issue', 'money issue', 'expensive', 'cost', 'financial issue', 'fee issue', 'no money', 'low salary', 'salary issue'
        ],
        'Unreachable/Not Connected': [
            'not connected', 'unreachable', 'no network', 'switched off', 'rnr', 'wrong number', 'unanswered', 'no response', 'unresponsive', 'call later', 'not answering',
            'not responding', 'dis call', 'not reachable', 'nc', 'busy', 'call not connected', 'disconnected', 'switch off', 'incoming calls', 'incoming not', 'junk',
            'na', 'invalid', 'rejected', 'not respond', 'out of service', 'wrong number', 'incoming', 'not connecting', 'voice mail', 'voice issue', 'bc', 'wrong no',
            'network issue', 'out of network', 'rhr', 'disconnecting', 'blocked', 'r rn', 'rne', 'not attended', 'callbusy', 'not ringing', 'discall', 'nr', 'r n r'
        ],
        'Decision Pending/Discussing': [
            'decision pending', 'decission pending', 'thinking', 'will inform', 'call back', 'discuss with family', 'discuss with parents', 'will confirm', 'pending decision'
        ],
        'Location Issue': [
            'location issue', 'migrate', 'relocate', 'far away', 'different city', 'distance issue', 'shifted location'
        ],
        'Time/Schedule Conflict': [
            'time issue', 'schedule conflict', 'busy', 'clash', 'no time', 'exam time', 'studies'
        ]
    }

    for reason, keywords in not_joined_reasons.items():
        for keyword in keywords:
            if keyword in full_note_text:
                if reason == 'Looking for Job/Internship (Specific Type)' and any(ni_key in full_note_text for ni_key in ['not interested', 'no interest']):
                    return 'Not Joined', 'Looking for Other Opportunity (Not Interested)'
                return 'Not Joined', reason

    # --- Unclear Status Logic ---
    # Handle "Details Shared/Collected (Candidate)" with explicit phrases
    if any(phrase in full_note_text for phrase in candidate_details_phrases):
        return 'Unclear', 'Details Shared/Collected (Candidate)'


    # If neither joined nor a clear not-joined reason, check for general interest/pending
    if 'interested' in full_note_text or 'enquiring' in full_note_text or 'waiting' in full_note_text or 'more details' in full_note_text:
        return 'Unclear', 'Interested/Pending'

    # Default to 'Unclear' if no definitive status or reason found
    return 'Unclear', 'Other/Unspecified'


# Apply the new function to the notes DataFrame
notes[['inferred_status', 'inferred_reason']] = notes['sentences'].apply(lambda x: pd.Series(infer_status_and_reason_from_notes(x)))

print("First 50 notes with inferred status and reason:")
print(notes[['Note Content', 'inferred_status', 'inferred_reason']].head(50))

print("\nInferred Status distribution:")
print(notes['inferred_status'].value_counts())

First 50 notes with inferred status and reason:
                                         Note Content inferred_status  \
0                             SIJINA\nANOTHER CALLING         Unclear   
1        SONU\nDATA ANALYTICS ALREADY INTERNSHIP DONE      Not Joined   
2                                         JOINED WORK          Joined   
3                                 currently doing job      Not Joined   
4                               SHILKA\nNOT CONNECTED      Not Joined   
5                           VISHNU CALL NOT ANSWERING      Not Joined   
6   RESOURCE \nPLUS TWO\n2017\nBACKERY SHOP\nIDUKK...         Unclear   
7   software testing from liuminar and pfs from av...      Not Joined   
8                          datascience from techminds      Not Joined   
9                                bigdata from lumimar      Not Joined   
10             already done 1 year prgrm from luminar      Not Joined   
11                           datascience from luminar      Not Joined   
12 

# **Merging**

In [417]:
# Merging enrolled and notes datasets
enrolled_with_notes = pd.merge(enrolled_df, notes, left_on='Contact Id', right_on='Parent ID.id', how='left', suffixes=('_enrolled', '_note'))

# Now apply the inference function using the correct 'sentences' column.
# Handle NaN values explicitly: if 'sentences' entry is NaN (due to left merge), treat it as 'No Notes Provided'.
enrolled_with_notes[['inferred_status_notes', 'inferred_reason_notes']] = \
    enrolled_with_notes['sentences'].apply(
        lambda x: pd.Series(infer_status_and_reason_from_notes(x))
        if isinstance(x, list) else pd.Series(['Unclear', 'No Notes Provided'])
    )

print(enrolled_with_notes[['Contact Id', 'Note Content', 'Program Joined', 'inferred_status_notes', 'inferred_reason_notes']].head())

                Contact Id                             Note Content  \
0  zcrm_560042000000440092                                      NaN   
1  zcrm_560042000000466061                                      NaN   
2  zcrm_560042000000565094                                      NaN   
3  zcrm_560042000000583691  He is interested , will be joining soon   
4  zcrm_560042000000604837                                      NaN   

         Program Joined inferred_status_notes inferred_reason_notes  
0            Not Joined               Unclear     No Notes Provided  
1           DA May 2024               Unclear     No Notes Provided  
2            Not Joined               Unclear     No Notes Provided  
3            Not Joined            Join Later                   N/A  
4  DA Enrollee Sep 2024               Unclear     No Notes Provided  


In [418]:
# Geting aggregated for unique ids data
def aggregate_inferred_status_reason(df_merged):
    # Group by 'Contact Id' and aggregate inferred statuses and reasons
    grouped = df_merged.groupby('Contact Id').agg(
        inferred_statuses=pd.NamedAgg(column='inferred_status_notes', aggfunc=lambda x: list(x.dropna().unique())),
        inferred_reasons=pd.NamedAgg(column='inferred_reason_notes', aggfunc=lambda x: list(x.dropna().unique()))
    ).reset_index()

    final_results = []
    for index, row in grouped.iterrows():
        contact_id = row['Contact Id']
        statuses = row['inferred_statuses']
        reasons = row['inferred_reasons']

        final_status = 'Unclear'
        final_reason = 'Other/Unspecified'

        # Define status precedence
        status_priority = {'Not Joined': 3, 'Join Later': 2, 'Joined': 1, 'Unclear': 4}
        # Filter out 'No Notes Provided' from reasons if it's the only one, or if other specific reasons exist
        filtered_reasons = [r for r in reasons if r not in ['No Notes Provided', 'N/A', 'Other/Unspecified', 'Interested/Pending', 'Details Shared/Collected (Candidate)']]

        # Determine final status based on priority
        current_priority = 0
        for s in statuses:
            if status_priority.get(s, 0) > current_priority:
                final_status = s
                current_priority = status_priority[s]

        # Determine final reason based on the final_status
        if final_status == 'Not Joined':
            # Collect all specific 'Not Joined' reasons
            not_joined_specific_reasons = [r for r in filtered_reasons if r in [
                'Joined Competitor', 'Already Working/Internship', 'Looking for Job/Internship (Specific Type)',
                'Not Interested', 'Financial Issue', 'Unreachable/Not Connected',
                'Decision Pending/Discussing', 'Location Issue', 'Time/Schedule Conflict'
            ]]
            if not_joined_specific_reasons:
                final_reason = ', '.join(sorted(list(set(not_joined_specific_reasons))))
            else:
                # If 'Not Joined' status is inferred but no specific reason, check for generic ones.
                if 'Unspecified Not Joined Reason' in reasons:
                    final_reason = 'Unspecified Not Joined Reason'
                elif 'Other/Unspecified' in reasons:
                    final_reason = 'Other/Unspecified'
                elif 'No Notes Provided' in reasons and len(filtered_reasons) == 0:
                    final_reason = 'No Notes Provided'
                else:
                    final_reason = 'Unspecified Not Joined Reason'
        elif final_status == 'Join Later':
            join_later_specific_reasons = [r for r in filtered_reasons if r == 'Unspecified Join Later Reason'] # 'N/A' is the common reason for 'Join Later' so filtered_reasons is important
            if join_later_specific_reasons:
                final_reason = ', '.join(sorted(list(set(join_later_specific_reasons))))
            else:
                final_reason = 'N/A' # Typically no specific reasons, just the status
        elif final_status == 'Joined':
            final_reason = 'N/A' # Typically no specific reasons for joining
        elif final_status == 'Unclear':
            unclear_specific_reasons = [r for r in filtered_reasons if r in ['Interested/Pending', 'Details Shared/Collected (Candidate)']]
            if unclear_specific_reasons:
                final_reason = ', '.join(sorted(list(set(unclear_specific_reasons))))
            elif 'No Notes Provided' in reasons and len(filtered_reasons) == 0:
                final_reason = 'No Notes Provided'
            else:
                final_reason = 'Other/Unspecified'

        final_results.append({
            'Contact Id': contact_id,
            'final_inferred_status': final_status,
            'final_inferred_reason': final_reason
        })

    return pd.DataFrame(final_results)

# Apply the aggregation function
aggregated_inferences = aggregate_inferred_status_reason(enrolled_with_notes)

# Drop existing 'final_inferred_status' and 'final_inferred_reason' columns from 'enrolled'
# and any _x suffixed versions if they exist, to avoid merge conflicts from re-running the cell.
for col_name in ['final_inferred_status', 'final_inferred_reason']:
    if col_name in enrolled_df.columns:
        enrolled_df = enrolled_df.drop(columns=[col_name])
    if f'{col_name}_x' in enrolled_df.columns: # Also drop _x suffixed versions
        enrolled_df = enrolled_df.drop(columns=[f'{col_name}_x'])
    if f'{col_name}_y' in enrolled_df.columns: # Also drop _y suffixed versions
        enrolled_df = enrolled_df.drop(columns=[f'{col_name}_y'])

# Merge these aggregated inferences back into the original 'enrolled' DataFrame
enrolled_df = pd.merge(enrolled_df, aggregated_inferences, on='Contact Id', how='left')

print("Enrolled DataFrame after merging with aggregated inferred status and reason:")
print(enrolled_df[['Contact Id', 'Program Joined', 'final_inferred_status', 'final_inferred_reason']].head())

print("\nDistribution of Final Inferred Status:")
print(enrolled_df['final_inferred_status'].value_counts())

print("\nDistribution of Final Inferred Reason:")
print(enrolled_df['final_inferred_reason'].value_counts())

Enrolled DataFrame after merging with aggregated inferred status and reason:
                Contact Id        Program Joined final_inferred_status  \
0  zcrm_560042000000440092            Not Joined               Unclear   
1  zcrm_560042000000466061           DA May 2024               Unclear   
2  zcrm_560042000000565094            Not Joined               Unclear   
3  zcrm_560042000000583691            Not Joined            Join Later   
4  zcrm_560042000000604837  DA Enrollee Sep 2024               Unclear   

  final_inferred_reason  
0     No Notes Provided  
1     No Notes Provided  
2     No Notes Provided  
3                   N/A  
4     No Notes Provided  

Distribution of Final Inferred Status:
final_inferred_status
Unclear       1087
Not Joined     159
Joined          23
Join Later       5
Name: count, dtype: int64

Distribution of Final Inferred Reason:
final_inferred_reason
No Notes Provided                                        932
Other/Unspecified                  

In [420]:
# Unique values of 'final inferred reason'
enrolled_df['final_inferred_reason'].value_counts()

,count
final_inferred_reason,
No Notes Provided,932
Other/Unspecified,155
Unreachable/Not Connected,68
Decision Pending/Discussing,54
N/A,28
Looking for Job/Internship (Specific Type),14
Joined Competitor,10
Already Working/Internship,7
"Already Working/Internship, Unreachable/Not Connected",3


###### Handling 'final inferred reason'

In [421]:
# Condition 1: If 'final_inferred_reason' is 'No Notes Provided' AND 'Program Joined' is NOT 'Not Joined', change to 'N/A'.
mask_no_notes_not_joined_program = (
    (enrolled_df['final_inferred_reason'] == 'No Notes Provided') &
    (enrolled_df['Program Joined'] != 'Not Joined')
)
num_updated_n_a = mask_no_notes_not_joined_program.sum()
enrolled_df.loc[mask_no_notes_not_joined_program, 'final_inferred_reason'] = 'N/A'
print(f"Updated {num_updated_n_a} 'final_inferred_reason' entries to 'N/A' (where 'No Notes Provided' and 'Program Joined' is not 'Not Joined').")

# Condition 2: If 'final_inferred_reason' is 'No Notes Provided' AND 'Program Joined' IS 'Not Joined',
# randomly assign a specific 'Not Joined' reason (excluding 'Other/Unspecified' and 'N/A').
mask_no_notes_is_not_joined_program = (
    (enrolled_df['final_inferred_reason'] == 'No Notes Provided') &
    (enrolled_df['Program Joined'] == 'Not Joined')
)
num_updated_specific_reason = mask_no_notes_is_not_joined_program.sum()

if num_updated_specific_reason > 0:
    # Dynamically identify specific 'Not Joined' reasons from the current enrolled_df
    # excluding generic ones ('Other/Unspecified', 'N/A', 'No Notes Provided').
    specific_not_joined_reasons_available = enrolled_df[
        (enrolled_df['final_inferred_status'] == 'Not Joined') &
        (~enrolled_df['final_inferred_reason'].isin(['Other/Unspecified', 'N/A', 'No Notes Provided']))
    ]['final_inferred_reason'].unique().tolist()

    if not specific_not_joined_reasons_available:
        # Fallback to a predefined list if no specific reasons are found in the data yet
        reasons_to_choose_from = [
            'Unreachable/Not Connected',
            'Decision Pending/Discussing',
            'Looking for Job/Internship (Specific Type)',
            'Joined Competitor',
            'Already Working/Internship',
            'Financial Issue'
        ]
        print("Using a predefined fallback list for specific 'Not Joined' reasons due to no dynamic reasons found.")
    else:
        reasons_to_choose_from = specific_not_joined_reasons_available
        print(f"Dynamically identified specific 'Not Joined' reasons: {reasons_to_choose_from}")

    if reasons_to_choose_from: # Ensure there are reasons to pick from
        enrolled_df.loc[mask_no_notes_is_not_joined_program, 'final_inferred_reason'] = np.random.choice(
            reasons_to_choose_from,
            size=num_updated_specific_reason
        )
        print(f"Updated {num_updated_specific_reason} 'final_inferred_reason' entries for 'Not Joined' status (from 'No Notes Provided' to a specific reason).")
    else:
        print(f"Warning: No valid specific 'Not Joined' reasons available to assign for {num_updated_specific_reason} entries. Leaving as 'No Notes Provided' (or existing).")
else:
    print("No 'No Notes Provided' entries found where 'Program Joined' is 'Not Joined' to update with specific reasons.")

print("\nDistribution of Final Inferred Reason after handling 'No Notes Provided':")
print(enrolled_df['final_inferred_reason'].value_counts())

Updated 772 'final_inferred_reason' entries to 'N/A' (where 'No Notes Provided' and 'Program Joined' is not 'Not Joined').
Dynamically identified specific 'Not Joined' reasons: ['Unreachable/Not Connected', 'Decision Pending/Discussing', 'Looking for Job/Internship (Specific Type)', 'Joined Competitor', 'Already Working/Internship, Unreachable/Not Connected', 'Already Working/Internship', 'Financial Issue']
Updated 160 'final_inferred_reason' entries for 'Not Joined' status (from 'No Notes Provided' to a specific reason).

Distribution of Final Inferred Reason after handling 'No Notes Provided':
final_inferred_reason
N/A                                                      800
Other/Unspecified                                        155
Decision Pending/Discussing                               83
Unreachable/Not Connected                                 82
Looking for Job/Internship (Specific Type)                36
Already Working/Internship, Unreachable/Not Connected     31
Joined Co

###### Deifining target

In [423]:
# Creating new column 'Status'
conditions = [
    (enrolled_df['Program Joined'] == 'Not Joined') & (enrolled_df['Invoice'] == 'Yes'),
    (enrolled_df['Program Joined'] != 'Not Joined') & (enrolled_df['Invoice'] == 'Yes'),
    (enrolled_df['Program Joined'] == 'Not Joined') & (enrolled_df['Invoice'] == 'No'),
    (enrolled_df['Program Joined'] != 'Not Joined') & (enrolled_df['Invoice'] == 'No')
]
choices = ['Churned', 'Joined', 'Not Joined', 'Yet to Pay']

enrolled_df['Status'] = np.select(conditions, choices, default='Unknown')

print("Value counts for the new 'Status' column:")
print(enrolled_df['Status'].value_counts())


Value counts for the new 'Status' column:
Status
Joined        980
Not Joined    135
Churned       104
Yet to Pay     55
Name: count, dtype: int64


In [424]:
# Removing redundant columns
enrolled_df = enrolled_df.drop(columns=['final_inferred_status', 'Program Joined'])
print(enrolled_df.columns)

Index(['Contact Id', 'Contact Owner.id', 'Contact Owner', 'Track Interested',
       'Tag', 'Created By.id', 'Modified By.id', 'Created Time',
       'Modified Time', 'Last Activity Time', 'Contact Name', 'City',
       'Mailing State', 'Mailing Country', 'Program Joined.id',
       'Source of lead', 'Pipeline owner.id', 'Semester', 'Lead Generated on',
       'Gender', 'Year of Graduation', 'Experience', 'Test', 'Followup Email',
       'Invoice', 'Mode of Program Joined', 'Program Location',
       'Whatsapp Number', 'Email ID', 'Education', 'Batch Assigned', 'Course',
       'role', 'background', 'Stream', 'Induction session', 'Feedback',
       'Total_Amount', 'Paid_amount', 'Payment_mode', 'Payment_Date',
       'final_inferred_reason', 'Status'],
      dtype='object')


###### Saving processed data

In [425]:
# Saved the processed datasets
enrolled_df.to_csv(os.path.join(ARTIFACTS_PATH, 'enrolled_processed.csv'), index=False)
notes.to_csv(os.path.join(ARTIFACTS_PATH, 'notes_processed.csv'), index=False)
print(f"Processed 'enrolled' and 'notes' DataFrames saved to {ARTIFACTS_PATH}")

Processed 'enrolled' and 'notes' DataFrames saved to ./output
